# Main 2015 Retraining Notebook (Rigorous CV + Stacked Optimization)

This notebook rebuilds the full training workflow with stronger optimization rigor:

- 8 models: Logistic Regression, KNN, Naive Bayes, Random Forest, XGBoost, CatBoost, AdaBoost, LightGBM
- Feature engineering, KNN imputation, standardization, and collinearity filtering (cutoff = 0.70)
- Leakage guard for BP-derived target (drops BP-like features when target can be BP-defined)
- Two-stage CV optimization and final training:
  - Stage 1: 5-fold CV, broad random search, proxy epochs = 25
  - Stage 2: 5-fold CV, local refinement + additional exploration, proxy epochs = 80
  - Final: candidate re-competition on validation, proxy epochs = 300 with early stopping where supported
- Calibration: Isotonic, Platt Scaling, Venn-Abers
- Calibration metrics: Expected Calibration Error (ECE) and Log Loss
- Stacking: OOF feature generation + meta-learner optimization + weighted-blend search
- Explainability: SHAP and LIME

In [24]:
# Optional one-time installs (uncomment if needed), then restart kernel.
# %pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib xgboost lightgbm catboost shap lime venn-abers

In [2]:
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import loguniform, randint, uniform

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

xgb_available = True
lgb_available = True
cat_available = True
shap_available = True
lime_available = True
venn_available = True
imblearn_available = True
smote_available = True
adasyn_available = True
torch_available = True
torch_cuda_available = False

try:
    from xgboost import XGBClassifier
except Exception:
    xgb_available = False
    XGBClassifier = None

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except Exception:
    lgb_available = False
    lgb = None
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    cat_available = False
    CatBoostClassifier = None

try:
    import shap
except Exception:
    shap_available = False
    shap = None

try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception:
    lime_available = False
    LimeTabularExplainer = None

try:
    from venn_abers import VennAbers
except Exception:
    venn_available = False
    VennAbers = None

try:
    from imblearn.over_sampling import ADASYN, SMOTE
except Exception:
    imblearn_available = False
    smote_available = False
    adasyn_available = False
    ADASYN = None
    SMOTE = None

try:
    import torch
    torch_cuda_available = bool(torch.cuda.is_available())
except Exception:
    torch_available = False
    torch = None

import joblib

print({
    "xgboost": xgb_available,
    "lightgbm": lgb_available,
    "catboost": cat_available,
    "shap": shap_available,
    "lime": lime_available,
    "venn_abers": venn_available,
    "imblearn": imblearn_available,
    "smote": smote_available,
    "adasyn": adasyn_available,
    "torch": torch_available,
    "torch_cuda": torch_cuda_available,
})

{'xgboost': True, 'lightgbm': True, 'catboost': True, 'shap': True, 'lime': True, 'venn_abers': True, 'imblearn': True, 'smote': True, 'adasyn': True, 'torch': False, 'torch_cuda': False}


In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "main_2015_balanced_gpu_artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
EXPLAIN_DIR = ARTIFACT_DIR / "explanations"
LIME_DIR = EXPLAIN_DIR / "lime"

for p in [ARTIFACT_DIR, MODEL_DIR, EXPLAIN_DIR, LIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET2015_CANDIDATES = [
    PROJECT_ROOT / "Datasets2015",
    PROJECT_ROOT.parent / "Datasets2015",
]


def _normalize_join_columns(df_in):
    rename_map = {}
    col_lc = {c.lower(): c for c in df_in.columns}
    for key in ["hhnum", "member_code"]:
        if key in col_lc and col_lc[key] != key:
            rename_map[col_lc[key]] = key
    return df_in.rename(columns=rename_map)


def _find_anthropometric_dataset_path():
    for base in DATASET2015_CANDIDATES:
        anthro_dir = base / "Anthropometric"
        if not anthro_dir.exists():
            continue

        csv_paths = sorted(
            [p for p in anthro_dir.glob("*.csv") if "dictionary" not in p.name.lower()]
        )
        preferred = [
            p
            for p in csv_paths
            if ("data-set" in p.name.lower()) or ("dataset" in p.name.lower())
        ]
        for p in preferred + csv_paths:
            return p
    return None


def _prepare_merged_with_anthro(base_path):
    if not base_path.exists():
        return None

    try:
        base_df = pd.read_csv(base_path)
    except Exception:
        return None

    anthro_tokens = ["weight", "height", "waist", "hip", "bmi", "whr"]
    has_anthro = any(any(tok in c.lower() for tok in anthro_tokens) for c in base_df.columns)
    if has_anthro:
        return None

    anthro_path = _find_anthropometric_dataset_path()
    if anthro_path is None:
        return None

    try:
        anthro_df = pd.read_csv(anthro_path)
    except Exception:
        return None

    base_df = _normalize_join_columns(base_df)
    anthro_df = _normalize_join_columns(anthro_df)

    join_keys = [k for k in ["hhnum", "member_code"] if k in base_df.columns and k in anthro_df.columns]
    if not join_keys:
        return None

    anthro_df = anthro_df.drop_duplicates(subset=join_keys, keep="first")

    overlap = [c for c in anthro_df.columns if c in base_df.columns and c not in join_keys]
    if overlap:
        anthro_df = anthro_df.rename(columns={c: f"{c}_anthro" for c in overlap})

    merged_df = base_df.merge(anthro_df, on=join_keys, how="left")
    out_path = PROJECT_ROOT / "merged_clinical_dietary_anthro_leftjoin.csv"
    merged_df.to_csv(out_path, index=False)

    print(f"Prepared anthropometric-augmented dataset: {out_path}")
    print(f"  source merged file: {base_path}")
    print(f"  source anthropometric file: {anthro_path}")
    print(f"  join keys: {join_keys}")
    return out_path


AUTO_MERGED_DATA_PATH = None
for merged_candidate in [
    PROJECT_ROOT / "merged_clinical_dietary_leftjoin.csv",
    PROJECT_ROOT.parent / "merged_clinical_dietary_leftjoin.csv",
]:
    AUTO_MERGED_DATA_PATH = _prepare_merged_with_anthro(merged_candidate)
    if AUTO_MERGED_DATA_PATH is not None:
        break

DATA_CANDIDATES = [
    AUTO_MERGED_DATA_PATH,
    PROJECT_ROOT / "merged_clinical_dietary_leftjoin.csv",
    PROJECT_ROOT.parent / "merged_clinical_dietary_leftjoin.csv",
    PROJECT_ROOT / "merged_clinical_leftjoin.csv",
    PROJECT_ROOT.parent / "merged_clinical_leftjoin.csv",
    PROJECT_ROOT / "Datasets" / "Clinical" / "clinical.csv",
    PROJECT_ROOT.parent / "Datasets" / "Clinical" / "clinical.csv",
]
DATA_CANDIDATES = [p for p in DATA_CANDIDATES if p is not None]

TARGET_CANDIDATES = ["hypertension", "htn", "target", "label", "outcome"]
ID_KEYWORDS = ["id", "subject", "participant", "seqn"]

COLLINEARITY_CUTOFF = 0.70
STAGE1_EPOCHS = 40
STAGE2_EPOCHS = 120
FINAL_EPOCHS = 500

STAGE1_TRIALS_PER_MODEL = 140
STAGE2_REFINEMENTS_PER_TOP_CONFIG = 28
TOP_K_STAGE1 = 10
TOP_K_STAGE2 = 4

CV_FOLDS_STAGE1 = 5
CV_FOLDS_STAGE2 = 6
STACK_OOF_FOLDS = 6
STACK_META_TRIALS = 400
STACK_BLEND_TRIALS = 7000
STACK_TOP_BASE_MODELS = 7

USE_GPU_WHEN_AVAILABLE = True

print(f"Artifacts: {ARTIFACT_DIR}")
print(
    {
        "stage1_trials_per_model": STAGE1_TRIALS_PER_MODEL,
        "stage2_refinements_per_top_config": STAGE2_REFINEMENTS_PER_TOP_CONFIG,
        "cv_folds": {"stage1": CV_FOLDS_STAGE1, "stage2": CV_FOLDS_STAGE2, "stack_oof": STACK_OOF_FOLDS},
        "stack_trials": {"meta": STACK_META_TRIALS, "blend": STACK_BLEND_TRIALS},
        "use_gpu_when_available": USE_GPU_WHEN_AVAILABLE,
        "torch_cuda": torch_cuda_available,
    }
)

Prepared anthropometric-augmented dataset: c:\Jon\College\Thesis\V2.2.1.1\merged_clinical_dietary_anthro_leftjoin.csv
  source merged file: c:\Jon\College\Thesis\V2.2.1.1\merged_clinical_dietary_leftjoin.csv
  source anthropometric file: c:\Jon\College\Thesis\V2.2.1.1\Datasets2015\Anthropometric\Jonathan Ralph_Baes_2026-03-26141834_data-set_anthrop.csv
  join keys: ['hhnum', 'member_code']
Artifacts: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts
{'stage1_trials_per_model': 140, 'stage2_refinements_per_top_config': 28, 'cv_folds': {'stage1': 5, 'stage2': 6, 'stack_oof': 6}, 'stack_trials': {'meta': 400, 'blend': 7000}, 'use_gpu_when_available': True, 'torch_cuda': False}


In [ ]:
# Force all dataset resolution to the local V2.2.1.1/Datasets2015 folder.
if Path.cwd().name.lower() == "v2.2.1.1":
    V2211_ROOT = Path.cwd()
elif (Path.cwd() / "V2.2.1.1").exists():
    V2211_ROOT = Path.cwd() / "V2.2.1.1"
elif Path.cwd().parent.name.lower() == "v2.2.1.1":
    V2211_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        f"Could not locate V2.2.1.1 directory from working directory: {Path.cwd()}"
    )

LOCAL_DATASET2015 = V2211_ROOT / "Datasets2015"
if not LOCAL_DATASET2015.exists():
    raise FileNotFoundError(
        f"Required local dataset folder not found: {LOCAL_DATASET2015}"
    )

PROJECT_ROOT = V2211_ROOT
ARTIFACT_DIR = PROJECT_ROOT / "main_2015_balanced_gpu_artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
EXPLAIN_DIR = ARTIFACT_DIR / "explanations"
LIME_DIR = EXPLAIN_DIR / "lime"
for p in [ARTIFACT_DIR, MODEL_DIR, EXPLAIN_DIR, LIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET2015_CANDIDATES = [LOCAL_DATASET2015]

AUTO_MERGED_DATA_PATH = None
for merged_candidate in [PROJECT_ROOT / "merged_clinical_dietary_leftjoin.csv"]:
    AUTO_MERGED_DATA_PATH = _prepare_merged_with_anthro(merged_candidate)
    if AUTO_MERGED_DATA_PATH is not None:
        break

DATA_CANDIDATES = [
    AUTO_MERGED_DATA_PATH,
    PROJECT_ROOT / "merged_clinical_dietary_leftjoin.csv",
    PROJECT_ROOT / "merged_clinical_leftjoin.csv",
    PROJECT_ROOT / "Datasets" / "Clinical" / "clinical.csv",
]
DATA_CANDIDATES = [p for p in DATA_CANDIDATES if p is not None]

print(f"Using V2.2.1.1 root: {PROJECT_ROOT}")
print(f"Using local Datasets2015: {LOCAL_DATASET2015}")
print("Data candidates (local only):")
for p in DATA_CANDIDATES:
    print(f"- {p}")

In [27]:
def resolve_data_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No dataset found. Checked: {candidates}")


def infer_target_column(df, target_candidates):
    lc = {c.lower(): c for c in df.columns}
    for t in target_candidates:
        if t.lower() in lc:
            return lc[t.lower()]

    # Fallback: partial match on candidate tokens.
    for c in df.columns:
        c_lc = c.lower()
        if any(t.lower() in c_lc for t in target_candidates):
            return c
    return None


def infer_bp_columns(df):
    sbp_aliases = ["ave_sbp", "sbp", "systolic", "sysbp", "s_bp"]
    dbp_aliases = ["ave_dbp", "dbp", "diastolic", "diabp", "d_bp"]

    lc = {c.lower(): c for c in df.columns}

    sbp_col = None
    for a in sbp_aliases:
        if a in lc:
            sbp_col = lc[a]
            break
    if sbp_col is None:
        for c in df.columns:
            c_lc = c.lower()
            if any(a in c_lc for a in sbp_aliases):
                sbp_col = c
                break

    dbp_col = None
    for a in dbp_aliases:
        if a in lc:
            dbp_col = lc[a]
            break
    if dbp_col is None:
        for c in df.columns:
            c_lc = c.lower()
            if any(a in c_lc for a in dbp_aliases):
                dbp_col = c
                break

    return sbp_col, dbp_col


def find_bp_like_columns(columns):
    bp_tokens = [
        "ave_sbp",
        "ave_dbp",
        "sbp",
        "dbp",
        "systolic",
        "diastolic",
        "sysbp",
        "diabp",
        "blood_pressure",
    ]
    cols = []
    for c in columns:
        c_lc = c.lower()
        if any(tok in c_lc for tok in bp_tokens):
            cols.append(c)
    return sorted(set(cols))


def find_dictionary_files(candidate_dirs):
    paths = []
    for d in candidate_dirs:
        if d.exists():
            paths.extend(d.rglob("*data-dictionary*.csv"))
            paths.extend(d.rglob("*data_dictionary*.csv"))

    uniq = []
    seen = set()
    for p in paths:
        key = str(p.resolve()).lower()
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return sorted(uniq)


def extract_identifier_variables_from_dictionaries(dict_paths):
    explicit_name_tokens = {
        "hhnum",
        "member_code",
        "seqn",
        "respondent_id",
        "participant_id",
        "subject_id",
        "person_id",
        "record_id",
        "case_id",
        "household_id",
        "hhid",
        "hh_id",
    }
    label_tokens = [
        "unique identifier",
        "unique household identifier",
        "merging variable",
        "household number",
        "member code",
        "respondent identifier",
        "participant identifier",
        "subject identifier",
        "person identifier",
        "record identifier",
        "case identifier",
    ]

    identifier_vars = set()
    for p in dict_paths:
        try:
            ddf = pd.read_csv(p, dtype=str, low_memory=False)
        except Exception:
            continue

        col_map = {c.lower(): c for c in ddf.columns}
        var_col = col_map.get("variable_name")
        if var_col is None:
            for cand in ["variable", "var_name", "varname", "column", "column_name"]:
                if cand in col_map:
                    var_col = col_map[cand]
                    break

        label_col = None
        for cand in ["variable_label", "var_label", "label", "description", "value_label", "definition"]:
            if cand in col_map:
                label_col = col_map[cand]
                break

        if var_col is None:
            continue

        vars_series = ddf[var_col].fillna("").astype(str).str.strip()
        if label_col is not None:
            labels_series = ddf[label_col].fillna("").astype(str).str.strip()
        else:
            labels_series = pd.Series([""] * len(vars_series), index=vars_series.index)

        for var_name, var_label in zip(vars_series, labels_series):
            if not var_name or var_name.lower() == "nan":
                continue

            v_lc = var_name.lower()
            l_lc = var_label.lower()
            name_is_id = (
                v_lc in explicit_name_tokens
                or v_lc == "id"
                or v_lc.startswith("id_")
                or v_lc.endswith("_id")
            )
            label_is_id = any(tok in l_lc for tok in label_tokens)
            if name_is_id or label_is_id:
                identifier_vars.add(var_name)

    return sorted(identifier_vars)


def find_first_column_case_insensitive(columns, candidates):
    lc = {c.lower(): c for c in columns}
    for cand in candidates:
        cand_lc = cand.lower()
        if cand_lc in lc:
            return lc[cand_lc]
    for c in columns:
        c_lc = c.lower()
        if any(cand.lower() in c_lc for cand in candidates):
            return c
    return None


def to_numeric_clean(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.where(~s.isin([9, 99, 888888, 999999]), np.nan)


def build_smoking_level_feature(df_in):
    used_cols = []

    smoking_level_col = find_first_column_case_insensitive(df_in.columns, ["smoking_level"])
    smoke_status_col = find_first_column_case_insensitive(df_in.columns, ["smoke_status"])
    current_smoking_col = find_first_column_case_insensitive(df_in.columns, ["current_smoking", "currentsmoking"])
    ever_smoke_col = find_first_column_case_insensitive(df_in.columns, ["ever_smk"])

    if smoking_level_col is not None:
        s = to_numeric_clean(df_in[smoking_level_col]).clip(lower=0, upper=3)
        used_cols.append(smoking_level_col)
        return s.astype(float), sorted(set(used_cols))

    idx = df_in.index
    smoke = pd.Series(np.nan, index=idx, dtype=float)

    if smoke_status_col is not None:
        status = to_numeric_clean(df_in[smoke_status_col])
        used_cols.append(smoke_status_col)

        smoke.loc[status == 0] = 0
        smoke.loc[status == 2] = 1
        smoke.loc[status == 1] = 2

        if current_smoking_col is not None:
            current = to_numeric_clean(df_in[current_smoking_col])
            used_cols.append(current_smoking_col)
            smoke.loc[(status == 1) & (current == 3)] = 3

        return smoke.astype(float), sorted(set(used_cols))

    if current_smoking_col is not None:
        current = to_numeric_clean(df_in[current_smoking_col])
        used_cols.append(current_smoking_col)

        smoke.loc[current == 0] = 0
        smoke.loc[current.isin([1, 2])] = 2
        smoke.loc[current == 3] = 3

        if ever_smoke_col is not None:
            ever = to_numeric_clean(df_in[ever_smoke_col])
            used_cols.append(ever_smoke_col)
            smoke.loc[(current == 0) & (ever > 0)] = 1

        return smoke.astype(float), sorted(set(used_cols))

    if ever_smoke_col is not None:
        ever = to_numeric_clean(df_in[ever_smoke_col])
        used_cols.append(ever_smoke_col)
        smoke.loc[ever == 0] = 0
        smoke.loc[ever > 0] = 1
        return smoke.astype(float), sorted(set(used_cols))

    return None, []


def build_alcohol_level_feature(df_in):
    used_cols = []

    alcohol_level_col = find_first_column_case_insensitive(df_in.columns, ["alcohol_level"])
    alcohol_status_col = find_first_column_case_insensitive(df_in.columns, ["alcohol_status"])
    alcohol_ever_col = find_first_column_case_insensitive(df_in.columns, ["alcohol"])
    current_alcohol_col = find_first_column_case_insensitive(df_in.columns, ["con_alcohol"])
    drink30_col = find_first_column_case_insensitive(df_in.columns, ["drnk_30days"])
    binge_col = find_first_column_case_insensitive(df_in.columns, ["binge_drink", "binge_drinking"])

    if alcohol_level_col is not None:
        a = to_numeric_clean(df_in[alcohol_level_col]).clip(lower=0, upper=3)
        used_cols.append(alcohol_level_col)
        return a.astype(float), sorted(set(used_cols))

    idx = df_in.index
    alcohol = pd.Series(np.nan, index=idx, dtype=float)

    if alcohol_status_col is not None:
        status = to_numeric_clean(df_in[alcohol_status_col])
        used_cols.append(alcohol_status_col)

        alcohol.loc[status == 0] = 0
        alcohol.loc[status == 2] = 1
        alcohol.loc[status == 1] = 2

        if binge_col is not None:
            binge = to_numeric_clean(df_in[binge_col])
            used_cols.append(binge_col)
            alcohol.loc[(status == 1) & (binge == 1)] = 3

        return alcohol.astype(float), sorted(set(used_cols))

    alcohol.loc[:] = 0

    if alcohol_ever_col is not None:
        ever = to_numeric_clean(df_in[alcohol_ever_col])
        used_cols.append(alcohol_ever_col)
        alcohol.loc[ever > 0] = 1

    if current_alcohol_col is not None:
        current = to_numeric_clean(df_in[current_alcohol_col])
        used_cols.append(current_alcohol_col)
        alcohol.loc[current == 1] = np.maximum(alcohol.loc[current == 1], 2)

    if drink30_col is not None:
        d30 = to_numeric_clean(df_in[drink30_col])
        used_cols.append(drink30_col)
        alcohol.loc[d30 == 1] = np.maximum(alcohol.loc[d30 == 1], 2)

    if binge_col is not None:
        binge = to_numeric_clean(df_in[binge_col])
        used_cols.append(binge_col)
        alcohol.loc[binge == 1] = 3

    if used_cols:
        alcohol = alcohol.astype(float)
        alcohol.loc[~alcohol.index.isin(alcohol.dropna().index)] = np.nan
        return alcohol, sorted(set(used_cols))

    return None, []


def build_bmi_feature(df_in):
    weight_col = find_first_column_case_insensitive(df_in.columns, ["weight"])
    height_col = find_first_column_case_insensitive(df_in.columns, ["height"])

    if weight_col is None or height_col is None:
        return None, []

    w = pd.to_numeric(df_in[weight_col], errors="coerce")
    h = pd.to_numeric(df_in[height_col], errors="coerce")

    h_m = h.copy()
    if pd.notna(h_m.median(skipna=True)) and float(h_m.median(skipna=True)) > 3.0:
        h_m = h_m / 100.0

    bmi = w / (h_m ** 2)
    bmi = bmi.replace([np.inf, -np.inf], np.nan)

    return bmi.astype(float), [weight_col, height_col]


def build_whr_feature(df_in):
    waist_col = find_first_column_case_insensitive(df_in.columns, ["waist"])
    hip_col = find_first_column_case_insensitive(df_in.columns, ["hip"])

    if waist_col is None or hip_col is None:
        return None, []

    waist = pd.to_numeric(df_in[waist_col], errors="coerce")
    hip = pd.to_numeric(df_in[hip_col], errors="coerce").replace(0, np.nan)

    whr = (waist / hip).replace([np.inf, -np.inf], np.nan)
    return whr.astype(float), [waist_col, hip_col]


data_path = resolve_data_path(DATA_CANDIDATES)
df = pd.read_csv(data_path)
print(f"Loaded data: {data_path}")
print(f"Shape: {df.shape}")

target_col = infer_target_column(df, TARGET_CANDIDATES)
TARGET_DEFINED_FROM_BP = False
TARGET_SOURCE_COLUMNS = []

if target_col is None:
    sbp_col, dbp_col = infer_bp_columns(df)
    if sbp_col is not None and dbp_col is not None:
        sbp = pd.to_numeric(df[sbp_col], errors="coerce")
        dbp = pd.to_numeric(df[dbp_col], errors="coerce")
        df["Hypertension"] = (((sbp >= 130) | (dbp >= 80)).fillna(False)).astype(int)
        target_col = "Hypertension"
        TARGET_DEFINED_FROM_BP = True
        TARGET_SOURCE_COLUMNS = [sbp_col, dbp_col]
        print(f"Target column created from: {sbp_col}, {dbp_col}")
    else:
        raise ValueError(
            f"Could not infer target column from {TARGET_CANDIDATES} and could not derive from BP columns. "
            f"Found BP-like columns: {[c for c in df.columns if 'bp' in c.lower()][:20]}"
        )

df = df.dropna(subset=[target_col]).copy()
y_raw = df[target_col]

if y_raw.nunique() != 2:
    raise ValueError(f"Target must be binary. Found {y_raw.nunique()} classes in '{target_col}'.")

if y_raw.dtype == "O":
    y = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=y_raw.index, name=target_col)
else:
    y = pd.Series(y_raw.astype(int), index=y_raw.index, name=target_col)

X = df.drop(columns=[target_col]).copy()
RAW_VARIABLES_BEFORE_CULL = sorted(X.columns.tolist())
print(f"\nVariables before culling: {len(RAW_VARIABLES_BEFORE_CULL)}")
for v in RAW_VARIABLES_BEFORE_CULL:
    print(v)

# Engineer smoking and alcohol features separately.
ENGINEERED_BASE_FEATURES_CREATED = []
ENGINEERED_BASE_FEATURE_SOURCE_MAP = {}

smoking_feature, smoking_sources = build_smoking_level_feature(X)
if smoking_feature is not None:
    X["fe_smoking_level"] = smoking_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("fe_smoking_level")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["fe_smoking_level"] = smoking_sources

alcohol_feature, alcohol_sources = build_alcohol_level_feature(X)
if alcohol_feature is not None:
    X["fe_alcohol_level"] = alcohol_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("fe_alcohol_level")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["fe_alcohol_level"] = alcohol_sources

# Engineer BMI and WHR, then remove source anthropometrics.
bmi_feature, bmi_source_cols = build_bmi_feature(X)
if bmi_feature is not None:
    X["bmi"] = bmi_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("bmi")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["bmi"] = bmi_source_cols

whr_feature, whr_source_cols = build_whr_feature(X)
if whr_feature is not None:
    X["whr"] = whr_feature
    ENGINEERED_BASE_FEATURES_CREATED.append("whr")
    ENGINEERED_BASE_FEATURE_SOURCE_MAP["whr"] = whr_source_cols

print(f"\nEngineered base features created: {ENGINEERED_BASE_FEATURES_CREATED}")
print(f"Engineered base feature sources: {ENGINEERED_BASE_FEATURE_SOURCE_MAP}")

BEHAVIOR_RAW_CANDIDATES = [
    "current_smoking",
    "currentsmoking",
    "ever_smk",
    "smoke_status",
    "smoking_level",
    "alcohol",
    "con_alcohol",
    "drnk_30days",
    "drnk_30d_num",
    "alcohol_status",
    "binge_drink",
    "binge_drinking",
    "alcohol_level",
]
x_lc = {c.lower(): c for c in X.columns}
BEHAVIOR_RAW_PURGED_COLUMNS = sorted({x_lc[c.lower()] for c in BEHAVIOR_RAW_CANDIDATES if c.lower() in x_lc})
if BEHAVIOR_RAW_PURGED_COLUMNS:
    X = X.drop(columns=BEHAVIOR_RAW_PURGED_COLUMNS, errors="ignore")
print(f"Purged smoking/alcohol raw columns: {BEHAVIOR_RAW_PURGED_COLUMNS}")

ANTHRO_SOURCE_DROPPED_COLUMNS = []
anthro_source_cols = sorted(set((bmi_source_cols or []) + (whr_source_cols or [])))
if anthro_source_cols:
    ANTHRO_SOURCE_DROPPED_COLUMNS = sorted({c for c in anthro_source_cols if c in X.columns and c.lower() not in {"bmi", "whr"}})
    if ANTHRO_SOURCE_DROPPED_COLUMNS:
        X = X.drop(columns=ANTHRO_SOURCE_DROPPED_COLUMNS, errors="ignore")

# Backward-compatible variable used by downstream audit cells.
HEIGHT_WEIGHT_DROPPED_COLUMNS = ANTHRO_SOURCE_DROPPED_COLUMNS.copy()
print(f"Dropped anthropometric source columns after BMI/WHR engineering: {HEIGHT_WEIGHT_DROPPED_COLUMNS}")

# Hard-drop known survey design and administrative columns.
MANUAL_NON_PREDICTIVE_CANDIDATES = [
    "regcode",
    "provcode",
    "provhuc",
    "psc",
    "csc",
    "rhc",
    "psurec",
    "strrec",
    "wgts",
    "fwgt",
    "finalwgt",
    "finalwgt1",
    "finalwgt4",
    "fwgth_natl_var",
    "fwgth_prov",
    "fwgth_natl2_var",
    "fwgti_natl_var",
    "fwgti_prov",
    "fwgti_natl2_var",
    "fwgti_prov2",
    "rep_natl",
    "rep_prov",
    "ms_psucode",
    "enns_year",
    "wrkplace",
    "interview_status",
    "intdate",
    "enumcode",
]
x_lc = {c.lower(): c for c in X.columns}
MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS = sorted({x_lc[c.lower()] for c in MANUAL_NON_PREDICTIVE_CANDIDATES if c.lower() in x_lc})
if MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS:
    X = X.drop(columns=MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS, errors="ignore")
print(f"Dropped manual non-predictive columns: {MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS}")

# Leakage guard: if target is BP-derived (or likely BP-defined), remove BP-like features from X.
BP_LEAKAGE_GUARD_ACTIVE = TARGET_DEFINED_FROM_BP or (target_col.lower() in {"hypertension", "htn"})
BP_DROPPED_COLUMNS = []
if BP_LEAKAGE_GUARD_ACTIVE:
    BP_DROPPED_COLUMNS = find_bp_like_columns(X.columns)
    BP_DROPPED_COLUMNS = sorted(set(BP_DROPPED_COLUMNS + TARGET_SOURCE_COLUMNS))
    X = X.drop(columns=BP_DROPPED_COLUMNS, errors="ignore")
    print(f"Dropped BP-like columns to prevent leakage: {BP_DROPPED_COLUMNS}")

# Dictionary-driven identifier drop (before split and before collinearity checks).
DICT_DIR_CANDIDATES = [
    PROJECT_ROOT / "Datasets",
    PROJECT_ROOT / "Datasets2015",
    PROJECT_ROOT.parent / "Datasets",
    PROJECT_ROOT.parent / "Datasets2015",
]
DICTIONARY_FILES_USED = find_dictionary_files(DICT_DIR_CANDIDATES)
DICT_IDENTIFIER_VARIABLES = extract_identifier_variables_from_dictionaries(DICTIONARY_FILES_USED)

x_lc = {c.lower(): c for c in X.columns}
DICT_IDENTIFIER_DROPPED_COLUMNS = sorted({x_lc[v.lower()] for v in DICT_IDENTIFIER_VARIABLES if v.lower() in x_lc})
if DICT_IDENTIFIER_DROPPED_COLUMNS:
    X = X.drop(columns=DICT_IDENTIFIER_DROPPED_COLUMNS, errors="ignore")
    print(f"Dropped dictionary-identified ID columns: {DICT_IDENTIFIER_DROPPED_COLUMNS}")
else:
    print("Dropped dictionary-identified ID columns: []")
print(f"Dictionary files scanned: {len(DICTIONARY_FILES_USED)}")

VARIABLES_AFTER_BASE_CULL = sorted(X.columns.tolist())
print(f"\nVariables after culling (before split): {len(VARIABLES_AFTER_BASE_CULL)}")
for v in VARIABLES_AFTER_BASE_CULL:
    print(v)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_tmp,
)

# Additional train-only fallback for ID/high-cardinality drops.
drop_id_like = []
for c in X_train.columns:
    c_lc = c.lower()
    if any(k in c_lc for k in ID_KEYWORDS):
        drop_id_like.append(c)
        continue
    nunique_ratio = X_train[c].nunique(dropna=False) / max(len(X_train), 1)
    if nunique_ratio > 0.995:
        drop_id_like.append(c)

X_train = X_train.drop(columns=drop_id_like, errors="ignore")
X_valid = X_valid.drop(columns=drop_id_like, errors="ignore")
X_test = X_test.drop(columns=drop_id_like, errors="ignore")

print(f"\nTarget column: {target_col}")
print(f"Dropped behavior raw columns: {len(BEHAVIOR_RAW_PURGED_COLUMNS)}")
print(f"Dropped anthropometric source columns: {len(HEIGHT_WEIGHT_DROPPED_COLUMNS)}")
print(f"Dropped manual non-predictive columns: {len(MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS)}")
print(f"Dropped dictionary ID columns: {len(DICT_IDENTIFIER_DROPPED_COLUMNS)}")
print(f"Dropped ID-like columns (train-only fallback): {len(drop_id_like)}")
print(f"Train/Valid/Test: {X_train.shape}, {X_valid.shape}, {X_test.shape}")

Loaded data: c:\Jon\College\Thesis\V2.2.1.1\merged_clinical_dietary_anthro_leftjoin.csv
Shape: (151189, 80)
Target column created from: Ave_SBP, Ave_DBP

Variables before culling: 80
Ave_DBP
Ave_SBP
Total_CHO
Total_Calcium
Total_Energy
Total_Fat
Total_FoodIntake
Total_Iron
Total_Niacin
Total_Protein
Total_Riboflavin
Total_Thiamin
Total_VitaminA
Total_VitaminC
age
age_anthro
agemos
alcohol
alcohol_status
anthro_group
binge_drinking
con_alcohol
csc
cu
current_smoking
drnk_30d_num
drnk_30days
ethnicity
ever_smk
fg1
fg10
fg11
fg12
fg13
fg14
fg15
fg16
fg17
fg18
fg19
fg2
fg20
fg21
fg23
fg24
fg25
fg26
fg27
fg3
fg4
fg5
fg6
fg7
fg8
fg9
finalwgt1
finalwgt4
fwgt
height
hhnum
hip
member_code
mos_lactation
mos_preg
provcode
provcode_anthro
psc
psc_anthro
psurec
psurec_anthro
regcode
regcode_anthro
sex
sex_anthro
smoke_status
strrec
strrec_anthro
waist
weight
wgts

Engineered base features created: ['fe_smoking_level', 'fe_alcohol_level', 'bmi', 'whr']
Engineered base feature sources: {'fe_smoking_l

In [28]:
# Fast raw-variable audit (before feature engineering/collinearity).
def _name_identifier_like(col_name):
    c = str(col_name).lower()
    tokens = ["id", "subject", "participant", "seqn", "identifier", "record", "uuid", "case", "sample", "hhnum", "member_code"]
    return any(t in c for t in tokens)


raw_vars = [c for c in df.columns if c != target_col]
raw_var_audit_df = pd.DataFrame({"variable": raw_vars})
raw_var_audit_df["identifier_like_name"] = raw_var_audit_df["variable"].apply(_name_identifier_like)
raw_var_audit_df["dropped_bp_guard"] = raw_var_audit_df["variable"].isin(BP_DROPPED_COLUMNS)
raw_var_audit_df["dropped_dict_rule"] = raw_var_audit_df["variable"].isin(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_manual_rule"] = raw_var_audit_df["variable"].isin(globals().get("MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_behavior_purge"] = raw_var_audit_df["variable"].isin(globals().get("BEHAVIOR_RAW_PURGED_COLUMNS", []))
raw_var_audit_df["dropped_height_weight_rule"] = raw_var_audit_df["variable"].isin(globals().get("HEIGHT_WEIGHT_DROPPED_COLUMNS", []))
raw_var_audit_df["dropped_fallback_id_rule"] = raw_var_audit_df["variable"].isin(drop_id_like)
raw_var_audit_df["kept_after_base_filters"] = raw_var_audit_df["variable"].isin(X_train.columns)

kept_identifier_like_df = raw_var_audit_df[
    (raw_var_audit_df["identifier_like_name"])
    & (raw_var_audit_df["kept_after_base_filters"])
] .copy()

raw_var_audit_path = ARTIFACT_DIR / "raw_variable_drop_audit_2015_rebuild.csv"
kept_identifier_like_path = ARTIFACT_DIR / "raw_variable_kept_identifier_like_2015_rebuild.csv"
raw_var_audit_df.to_csv(raw_var_audit_path, index=False)
kept_identifier_like_df.to_csv(kept_identifier_like_path, index=False)

print(f"Saved: {raw_var_audit_path}")
print(f"Saved: {kept_identifier_like_path}")
print("Engineered base features:", globals().get("ENGINEERED_BASE_FEATURES_CREATED", []))
print("Drop summary:")
print(
    {
        "raw_variables": int(len(raw_var_audit_df)),
        "dropped_bp_guard": int(raw_var_audit_df['dropped_bp_guard'].sum()),
        "dropped_dict_rule": int(raw_var_audit_df['dropped_dict_rule'].sum()),
        "dropped_manual_rule": int(raw_var_audit_df['dropped_manual_rule'].sum()),
        "dropped_behavior_purge": int(raw_var_audit_df['dropped_behavior_purge'].sum()),
        "dropped_height_weight_rule": int(raw_var_audit_df['dropped_height_weight_rule'].sum()),
        "dropped_fallback_id_rule": int(raw_var_audit_df['dropped_fallback_id_rule'].sum()),
        "kept_identifier_like": int(len(kept_identifier_like_df)),
    }
)
kept_identifier_like_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\raw_variable_drop_audit_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\raw_variable_kept_identifier_like_2015_rebuild.csv
Engineered base features: ['fe_smoking_level', 'fe_alcohol_level', 'bmi', 'whr']
Drop summary:
{'raw_variables': 80, 'dropped_bp_guard': 2, 'dropped_dict_rule': 2, 'dropped_manual_rule': 10, 'dropped_behavior_purge': 9, 'dropped_height_weight_rule': 4, 'dropped_fallback_id_rule': 0, 'kept_identifier_like': 0}


,variable,identifier_like_name,dropped_bp_guard,dropped_dict_rule,dropped_manual_rule,dropped_behavior_purge,dropped_height_weight_rule,dropped_fallback_id_rule,kept_after_base_filters


In [29]:
def find_first_column(cols, keys):
    low = {c.lower(): c for c in cols}
    for k in keys:
        for c_lc, c in low.items():
            if k in c_lc:
                return c
    return None


def feature_engineering(df_in):
    df_fe = df_in.copy()
    engineered_added = []

    age_col = find_first_column(df_fe.columns, ["age", "agemos"])
    bmi_col = find_first_column(df_fe.columns, ["bmi"])
    chol_col = find_first_column(df_fe.columns, ["chol"])
    hdl_col = find_first_column(df_fe.columns, ["hdl"])
    waist_col = find_first_column(df_fe.columns, ["waist"])
    hip_col = find_first_column(df_fe.columns, ["hip"])
    energy_col = find_first_column(df_fe.columns, ["energy", "kcal"])
    fat_col = find_first_column(df_fe.columns, ["total_fat", "fat"])

    if age_col and bmi_col:
        df_fe["fe_age_x_bmi"] = df_fe[age_col] * df_fe[bmi_col]
        engineered_added.append("fe_age_x_bmi")

    if chol_col and hdl_col:
        den = df_fe[hdl_col].replace(0, np.nan)
        df_fe["fe_chol_hdl_ratio"] = df_fe[chol_col] / den
        engineered_added.append("fe_chol_hdl_ratio")

    if waist_col and hip_col:
        den = df_fe[hip_col].replace(0, np.nan)
        df_fe["fe_waist_hip_ratio"] = df_fe[waist_col] / den
        engineered_added.append("fe_waist_hip_ratio")

    if energy_col and fat_col:
        den = df_fe[energy_col].replace(0, np.nan)
        df_fe["fe_fat_density"] = df_fe[fat_col] / den
        engineered_added.append("fe_fat_density")

    return df_fe, sorted(set(engineered_added))


X_train_fe, engineered_train = feature_engineering(X_train)
X_valid_fe, engineered_valid = feature_engineering(X_valid)
X_test_fe, engineered_test = feature_engineering(X_test)

all_cols = sorted(set(X_train_fe.columns) | set(X_valid_fe.columns) | set(X_test_fe.columns))
X_train_fe = X_train_fe.reindex(columns=all_cols)
X_valid_fe = X_valid_fe.reindex(columns=all_cols)
X_test_fe = X_test_fe.reindex(columns=all_cols)

ENGINEERED_FEATURES_PHASE2 = sorted(set(engineered_train) | set(engineered_valid) | set(engineered_test))
print(f"Feature counts after engineering: {X_train_fe.shape[1]}")
print("Phase-2 engineered features:", ENGINEERED_FEATURES_PHASE2)

Feature counts after engineering: 59
Phase-2 engineered features: ['fe_age_x_bmi', 'fe_fat_density']


In [30]:
X_train_fe.describe()

,Total_CHO,Total_Calcium,Total_Energy,Total_Fat,Total_FoodIntake,Total_Iron,Total_Niacin,Total_Protein,Total_Riboflavin,Total_Thiamin,Total_VitaminA,Total_VitaminC,age,age_anthro,agemos,anthro_group,bmi,cu,ethnicity,fe_age_x_bmi,fe_alcohol_level,fe_fat_density,fe_smoking_level,fg1,fg10,fg11,fg12,fg13,fg14,fg15,fg16,fg17,fg18,fg19,fg2,fg20,fg21,fg23,fg24,fg25,fg26,fg27,fg3,fg4,fg5,fg6,fg7,fg8,fg9,mos_lactation,mos_preg,provcode_anthro,psc_anthro,psurec_anthro,regcode_anthro,sex,sex_anthro,strrec_anthro,whr
count,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,105832.000000,105832.000000,105832.000000,105832.000000,105360.000000,26632.000000,105832.000000,105360.000000,21439.000000,26632.000000,21508.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,26632.000000,2767.000000,1053.000000,105832.000000,105832.000000,105832.000000,105832.000000,105832.000000,105832.000000,105832.000000,85197.000000
mean,1607.702395,45.436596,9050.155086,170.502139,4021.191006,1964.297185,89.407654,268.304206,3.340063,3.989993,2090.041617,216.523177,30.792238,30.792238,369.506855,3.501096,20.581350,5.368160,0.063525,686.248316,0.915901,0.018501,0.592989,1803.295577,407.913482,170.295439,26.130729,144.164710,808.695617,451.178002,242.319226,115.198389,78.579381,179.361573,1556.335385,144.974132,34.387441,67.912263,122.196366,71.163833,44.622193,6.410340,136.164631,110.795560,70.551733,58.387921,39.820271,622.094864,214.181382,11.138778,5.698955,40.889684,47.866279,4.453029,12.630594,1.517131,1.517131,307.927357,0.865178
std,863.339120,26.781527,4590.014890,152.818796,2303.951065,1860.025292,52.305876,145.542185,2.680020,2.824709,5225.308101,269.486528,20.815613,20.815613,249.787355,1.206272,4.972710,2.423017,0.245181,521.824689,1.086676,0.011287,1.065638,1014.161686,701.885898,736.188197,153.078234,717.004068,738.256291,535.477373,440.314320,314.539900,121.365095,547.604845,1010.845136,413.182077,342.543374,128.741090,197.023447,172.986236,58.724649,58.829078,520.089468,173.521919,338.215809,96.458694,90.123325,853.668369,399.135141,0.856414,2.092802,22.507968,49.412112,8.582027,11.901049,0.499709,0.499709,175.750512,0.071155
min,32.910000,1.190020,292.459991,0.581470,79.649994,48.730000,3.286000,7.009060,0.065510,0.082105,0.000000,0.000000,3.010000,3.010000,36.119999,1.000000,6.173864,0.500000,0.000000,27.374839,0.000000,0.001250,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.400978
25%,1022.560059,28.129999,5942.060059,70.220001,2582.050049,1075.177551,54.182190,171.850281,1.910019,2.220000,642.399963,54.270000,12.510000,12.510000,150.119995,3.000000,16.505701,4.000000,0.000000,209.647583,0.000000,0.009931,0.000000,1101.574982,35.400002,0.000000,0.000000,0.000000,300.070007,65.629997,0.000000,0.000000,0.000000,0.000000,900.700012,0.000000,0.000000,11.900000,33.799999,4.000000,11.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,118.400002,0.000000,10.000000,4.000000,22.000000,0.000000,2.000000,5.000000,1.000000,1.000000,155.000000,0.814907
50%,1463.383057,40.009998,8284.234863,129.834549,3628.559937,1580.420044,79.307999,241.050003,2.787200,3.310000,1161.859985,143.020004,25.940000,25.940000,311.279999,4.000000,20.175747,5.000000,0.000000,570.521947,0.000000,0.016557,0.000000,1630.729981,205.100006,0.000000,0.000000,0.000000,647.099976,307.630005,0.000000,0.000000,0.000000,0.000000

In [31]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


num_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train_fe.columns if c not in num_cols]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", KNNImputer(n_neighbors=5)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

X_train_proc = preprocessor.fit_transform(X_train_fe)
X_valid_proc = preprocessor.transform(X_valid_fe)
X_test_proc = preprocessor.transform(X_test_fe)

feature_names = preprocessor.get_feature_names_out()

X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train_fe.index)
X_valid_proc = pd.DataFrame(X_valid_proc, columns=feature_names, index=X_valid_fe.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test_fe.index)

COLLINEARITY_PROTECTED_FEATURES = sorted(
    [c for c in ["num__bmi", "num__whr"] if c in X_train_proc.columns]
)


def collinearity_filter(df_in, cutoff=0.70, protected_cols=None):
    protected = set(protected_cols or [])
    corr = df_in.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    drop_cols = []
    for c in upper.columns:
        if c in protected:
            continue
        if (upper[c] > cutoff).any():
            drop_cols.append(c)

    keep_cols = [c for c in df_in.columns if c not in drop_cols]
    return keep_cols, drop_cols


keep_cols, dropped_cols = collinearity_filter(
    X_train_proc,
    cutoff=COLLINEARITY_CUTOFF,
    protected_cols=COLLINEARITY_PROTECTED_FEATURES,
)

X_train_final = X_train_proc[keep_cols].copy()
X_valid_final = X_valid_proc[keep_cols].copy()
X_test_final = X_test_proc[keep_cols].copy()

if "BP_LEAKAGE_GUARD_ACTIVE" in globals() and BP_LEAKAGE_GUARD_ACTIVE:
    bp_tokens_guard = ["ave_sbp", "ave_dbp", "sbp", "dbp", "systolic", "diastolic", "sysbp", "diabp", "blood_pressure"]
    leaked_bp_cols = [c for c in X_train_final.columns if any(tok in c.lower() for tok in bp_tokens_guard)]
    if leaked_bp_cols:
        raise ValueError(f"Leakage guard failed: BP-like features still present in model matrix: {leaked_bp_cols}")
    print("BP leakage guard: passed (no BP-like features in final matrix)")

print(f"Processed feature count: {len(feature_names)}")
print(f"Protected from collinearity dropping: {COLLINEARITY_PROTECTED_FEATURES}")
print(f"Dropped for collinearity (>{COLLINEARITY_CUTOFF}): {len(dropped_cols)}")
print(f"Final feature count: {X_train_final.shape[1]}")

BP leakage guard: passed (no BP-like features in final matrix)
Processed feature count: 59
Protected from collinearity dropping: ['num__bmi', 'num__whr']
Dropped for collinearity (>0.7): 20
Final feature count: 39


In [32]:
# Sanity checks for requested variable cleanup and feature engineering.
requested_remove = ["provcode", "psc", "psurec", "regcode", "strrec", "wgts", "weight", "height", "hip", "waist"]

print("Manual non-predictive dropped columns:")
print(sorted(globals().get("MANUAL_NON_PREDICTIVE_DROPPED_COLUMNS", [])))

print("\nBehavior raw columns purged:")
print(sorted(globals().get("BEHAVIOR_RAW_PURGED_COLUMNS", [])))

print("\nAnthropometric source columns dropped after BMI/WHR engineering:")
print(sorted(globals().get("HEIGHT_WEIGHT_DROPPED_COLUMNS", [])))

print("\nEngineered base features created:")
print(globals().get("ENGINEERED_BASE_FEATURES_CREATED", []))
print(globals().get("ENGINEERED_BASE_FEATURE_SOURCE_MAP", {}))

print("\nSeparate engineered behavior features present:")
print("  fe_smoking_level:", "fe_smoking_level" in X_train.columns)
print("  fe_alcohol_level:", "fe_alcohol_level" in X_train.columns)
print("  bmi:", "bmi" in X_train.columns)
print("  whr:", "whr" in X_train.columns)

fe_log1p_cols = [c for c in X_train_fe.columns if c.startswith("fe_log1p_")]
print("\nfe_log1p engineered columns count:", len(fe_log1p_cols))
print("Phase-2 engineered features:", globals().get("ENGINEERED_FEATURES_PHASE2", []))

print("\nCollinearity protected processed features:", globals().get("COLLINEARITY_PROTECTED_FEATURES", []))

print("\nRequested remove status in base train features:")
for c in requested_remove:
    print(f"  {c}:", c in X_train.columns)

final_lower = [c.lower() for c in X_train_final.columns]
print("\nRequested remove tokens in final model matrix names:")
for c in requested_remove:
    has_token = any(c in name for name in final_lower)
    print(f"  {c}:", has_token)

Manual non-predictive dropped columns:
['csc', 'finalwgt1', 'finalwgt4', 'fwgt', 'provcode', 'psc', 'psurec', 'regcode', 'strrec', 'wgts']

Behavior raw columns purged:
['alcohol', 'alcohol_status', 'binge_drinking', 'con_alcohol', 'current_smoking', 'drnk_30d_num', 'drnk_30days', 'ever_smk', 'smoke_status']

Anthropometric source columns dropped after BMI/WHR engineering:
['height', 'hip', 'waist', 'weight']

Engineered base features created:
['fe_smoking_level', 'fe_alcohol_level', 'bmi', 'whr']
{'fe_smoking_level': ['current_smoking', 'smoke_status'], 'fe_alcohol_level': ['alcohol_status', 'binge_drinking'], 'bmi': ['weight', 'height'], 'whr': ['waist', 'hip']}

Separate engineered behavior features present:
  fe_smoking_level: True
  fe_alcohol_level: True
  bmi: True
  whr: True

fe_log1p engineered columns count: 0
Phase-2 engineered features: ['fe_age_x_bmi', 'fe_fat_density']

Collinearity protected processed features: ['num__bmi', 'num__whr']

Requested remove status in base t

In [33]:
# Diagnose anthropometric source availability for BMI/WHR engineering.
anthro_cols = [c for c in df.columns if any(tok in c.lower() for tok in ["weight", "height", "bmi", "waist", "hip", "whr"])]
print("Anthropometric columns found in raw merged dataframe:", anthro_cols)

x_train_cols = [c for c in X_train.columns if any(tok in c.lower() for tok in ["weight", "height", "bmi", "waist", "hip", "whr"])]
print("Anthropometric columns retained in X_train:", x_train_cols)

Anthropometric columns found in raw merged dataframe: ['weight', 'waist', 'hip', 'height']
Anthropometric columns retained in X_train: ['bmi', 'whr']


In [34]:
# Variable audit: raw -> engineered -> processed -> final model matrix.
def _is_identifier_like(name):
    n = str(name).lower()
    id_tokens = ["member_code", "hhnum"]
    return any(tok in n for tok in id_tokens)


def _is_bp_like(name):
    n = str(name).lower()
    bp_tokens = ["ave_sbp", "ave_dbp"]
    return any(tok in n for tok in bp_tokens)


dict_drop_set = set(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
base_predictor_cols = [c for c in df.columns if c != target_col]
train_base_before_id_drop = X.loc[y_train.index].copy()
nunique_ratio_map = {
    c: train_base_before_id_drop[c].nunique(dropna=False) / max(len(train_base_before_id_drop), 1)
    for c in train_base_before_id_drop.columns
}

audit_rows = []
for col in base_predictor_cols:
    in_bp_drop = col in BP_DROPPED_COLUMNS
    in_dict_drop = col in dict_drop_set
    in_id_drop = col in drop_id_like
    kept_after_base_filters = col in X_train.columns
    uniq_ratio = nunique_ratio_map.get(col, np.nan)

    review_bits = []
    if in_dict_drop:
        review_bits.append("dropped by dictionary identifier rule")
    if in_id_drop:
        review_bits.append("dropped by train-only fallback rule")
    if _is_identifier_like(col) and kept_after_base_filters and not in_dict_drop and not in_id_drop:
        review_bits.append("identifier-like retained in base features")
    if _is_bp_like(col) and kept_after_base_filters and BP_LEAKAGE_GUARD_ACTIVE:
        review_bits.append("bp-like retained while BP guard active")

    audit_rows.append(
        {
            "stage": "raw_base",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": uniq_ratio,
            "dropped_bp_guard": in_bp_drop,
            "dropped_dict_rule": in_dict_drop,
            "dropped_id_rule": in_id_drop,
            "kept_after_base_filters": kept_after_base_filters,
            "review_flag": "; ".join(review_bits),
        }
    )

engineered_only = sorted(set(X_train_fe.columns) - set(X_train.columns))
for col in engineered_only:
    audit_rows.append(
        {
            "stage": "engineered",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": col in X_train_fe.columns,
            "review_flag": "",
        }
    )

for col in feature_names:
    audit_rows.append(
        {
            "stage": "processed",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": col in keep_cols,
            "review_flag": "dropped by collinearity" if col in dropped_cols else "",
        }
    )

final_var = X_train_final.var(numeric_only=True)
for col in X_train_final.columns:
    near_zero_var = bool(final_var.get(col, np.nan) <= 1e-10) if pd.notna(final_var.get(col, np.nan)) else False
    review_bits = []
    if _is_identifier_like(col):
        review_bits.append("identifier-like feature in final matrix")
    if _is_bp_like(col) and BP_LEAKAGE_GUARD_ACTIVE:
        review_bits.append("bp-like feature in final matrix")
    if near_zero_var:
        review_bits.append("near-zero variance in final matrix")

    audit_rows.append(
        {
            "stage": "final_model",
            "variable": col,
            "identifier_like": _is_identifier_like(col),
            "bp_like": _is_bp_like(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
            "review_flag": "; ".join(review_bits),
        }
    )

variable_audit_df = pd.DataFrame(audit_rows)
variable_audit_df = variable_audit_df.sort_values(["stage", "variable"]).reset_index(drop=True)

suspect_kept_df = variable_audit_df[
    (variable_audit_df["stage"] == "final_model")
    & (variable_audit_df["review_flag"].astype(str).str.len() > 0)
] .copy()

runtime_var_names = sorted([name for name in globals().keys() if not name.startswith("_")])
runtime_var_df = pd.DataFrame({"name": runtime_var_names})
runtime_var_df["likely_temporary"] = runtime_var_df["name"].apply(
    lambda n: (len(n) <= 2 and n.islower()) or n in {"rows", "params", "met", "i", "c", "m", "p"}
 )

var_audit_path = ARTIFACT_DIR / "variable_audit_2015_rebuild.csv"
suspect_path = ARTIFACT_DIR / "variable_audit_suspect_kept_2015_rebuild.csv"
runtime_path = ARTIFACT_DIR / "runtime_variable_list_2015_rebuild.csv"

variable_audit_df.to_csv(var_audit_path, index=False)
suspect_kept_df.to_csv(suspect_path, index=False)
runtime_var_df.to_csv(runtime_path, index=False)

print(f"Saved: {var_audit_path}")
print(f"Saved: {suspect_path}")
print(f"Saved: {runtime_path}")
print(f"Dictionary files scanned: {len(globals().get('DICTIONARY_FILES_USED', []))}")
print(f"Dictionary ID columns dropped: {sorted(list(dict_drop_set))}")
print("\nAudit counts by stage:")
print(variable_audit_df.groupby("stage")["variable"].count().to_string())
print("\nPotentially suspect kept final features:", len(suspect_kept_df))
suspect_kept_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\variable_audit_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\variable_audit_suspect_kept_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\runtime_variable_list_2015_rebuild.csv
Dictionary files scanned: 10
Dictionary ID columns dropped: ['hhnum', 'member_code']

Audit counts by stage:
stage
engineered      2
final_model    39
processed      59
raw_base       80

Potentially suspect kept final features: 0


,stage,variable,identifier_like,bp_like,train_nunique_ratio,dropped_bp_guard,dropped_dict_rule,dropped_id_rule,kept_after_base_filters,review_flag


In [35]:
X_train_final

,num__Total_CHO,num__Total_Calcium,num__Total_Fat,num__Total_Iron,num__Total_Riboflavin,num__Total_VitaminC,num__age,num__anthro_group,num__bmi,num__cu,num__ethnicity,num__fe_alcohol_level,num__fe_smoking_level,num__fg10,num__fg11,num__fg12,num__fg15,num__fg16,num__fg17,num__fg18,num__fg19,num__fg21,num__fg23,num__fg24,num__fg26,num__fg27,num__fg3,num__fg4,num__fg5,num__fg6,num__fg7,num__fg9,num__mos_lactation,num__mos_preg,num__provcode_anthro,num__psc_anthro,num__psurec_anthro,num__regcode_anthro,num__whr
139444,-0.472025,-0.125358,-0.046225,-0.240157,-0.112022,-0.574963,0.387104,0.413593,0.631213,-0.102038,-0.259096,-0.199548,-0.383205,-0.458573,-0.311323,-0.261006,2.046242,0.102747,-0.571813,0.647736,0.422206,0.461767,-0.312164,-0.299264,0.387675,-0.059171,0.679482,-0.168732,-0.313806,-0.449085,0.917642,0.015189,1.803980,0.319604,-0.172814,-0.968720,-0.052788,-0.389093,-0.305340
40637,-0.072289,0.065960,0.478650,0.473679,0.889000,0.599103,-0.916251,-0.415411,-0.802097,0.061608,-0.259096,-0.199548,-0.643735,-0.419416,0.082312,0.160818,-0.442835,0.858912,0.475200,0.631419,1.181203,-0.162103,-0.367825,1.587667,-0.266458,0.089653,-0.358155,-0.348430,0.184037,0.266420,1.521834,-0.412404,-0.782996,-0.522737,1.515485,-0.968720,1.228966,0.031040,-0.101507
139758,0.336680,0.079086,-0.528802,2.765753,-0.119193,0.396061,1.091866,0.413593,0.428059,-0.347507,-0.259096,0.510805,0.658914,-0.602822,-0.354362,-0.261006,1.398937,-0.479527,-0.571813,0.158587,0.204034,-0.162103,-0.430956,-0.606131,-0.588525,-0.171057,-0.374775,-0.441711,3.465435,-0.759491,-0.509514,0.102799,0.510492,2.004286,-0.172814,1.034847,-0.402358,-0.389093,0.155540
35179,0.272490,1.340504,1.739803,0.113967,0.973761,0.355328,-0.519912,0.413593,-0.141027,-0.429330,-0.259096,0.510805,-0.122675,0.291771,0.282366,1.900321,0.003444,3.062995,-0.275433,1.406072,-0.121126,-0.008535,0.730937,0.238259,0.242429,-0.171057,-0.239828,1.043798,-0.313806,0.375637,-0.336342,1.050847,-0.351833,0.151136,1.471056,-0.968720,3.792475,0.031040,-1.367427
19784,-1.695579,-1.737968,-1.108524,-1.020727,-1.228547,-1.239939,-1.024344,-1.244415,-1.790892,-0.838445,-0.259096,-0.909902,-0.643735,-0.863088,-0.354362,-0.261006,-0.663692,0.017080,-0.571813,-1.010754,-0.517298,-0.162103,-0.664443,-0.744712,-1.191611,-0.171057,-0.374775,-0.488128,-0.313806,-0.895029,-0.691014,-0.814428,-0.782996,-1.702014,-0.750390,1.034847,0.063735,-0.137013,0.104614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101505,0.350755,-0.183245,-0.044190,-0.204688,-0.041697,-0.191548,1.274423,0.413593,1.070634,0.470723,-0.259096,0.274021,-0.383205,-0.484453,0.242217,-0.261006,0.017711,-0.062802,0.709865,1.434231,-0.283969,-0.162103,-0.075545,0.407791,1.982749,-0.171057,-0.374775,-0.412282,-0.033002,-0.499307,-0.550811,-0.047648,1.372818,0.151136,0.493619,-0.968720,-0.169312,2.467811,-0.292752
109089,-0.411139,-0.492341,-0.413627,0.316012,0.565743,-0.189379,1.357534,0.413593,1.777777,-0.490697,-0.259096,0.274021,-0.643735,-0.594510,-0.025609,0.408476,0.753434,-0.835442,-0.571813,-0.055796,2.607949,0.183425,-0.235492,-0.445465,-0.328030,-0.082191,0.024269,0.048148,-0.313806,-0.014941,-0.225117,-0.468790,0.941655,1.330414,-0.928106,-0.968720,-0.169312,-0.641173,-0.085378
62950,-0.380454,-0.717048,1.644523,-0.637531,-0.001933,-0.434277,1.191312,0.413593,-0.220768,-0.347507,-0.259096,0.984375,-0.122675,-0.299931,-0.073587,-0.261006,-0.464873,1.832925,-0.571813,1.200059,-0.372858,-0.162103,-0.375006,0.659677,-0.627731,-0.171057,-0.374775,-0.878136,-0.313806,0.854920,-0.544565,-0.391364,1.372818,-0.691205,1.204482,1.034847,-0.169312,0.283120,-0.345121
18640,0.414805,0.025398,-0.662599,0.793855,-0.301913,-0.063850,-0.930183,-0.415411,-0.528145,0.634369,-0.259096,-0.436333,-0.643735,-0.225687,-0.354362,-0.261006,0.044857,-0.312882,-0.182750,-1.010754,-0.212656,-0.162103,-0.570952,-0.458530,0.006141,-0.171057,0.999100,-0.32431

In [36]:
MODEL_SPACES = {
    "log_reg": {
        "C": loguniform(1e-4, 5e2),
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
    },
    "knn": {
        "n_neighbors": randint(3, 121),
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    },
    "naive_bayes": {
        "var_smoothing": loguniform(1e-12, 1e-6),
    },
    "random_forest": {
        "max_depth": [None, 4, 6, 8, 12, 16, 20],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    },
    "xgboost": {
        "learning_rate": uniform(0.01, 0.29),
        "max_depth": randint(3, 10),
        "subsample": uniform(0.55, 0.45),
        "colsample_bytree": uniform(0.55, 0.45),
        "min_child_weight": randint(1, 12),
        "gamma": uniform(0.0, 3.0),
        "reg_lambda": loguniform(1e-3, 30),
    },
    "catboost": {
        "learning_rate": uniform(0.01, 0.29),
        "depth": randint(4, 11),
        "l2_leaf_reg": loguniform(1.0, 20.0),
        "random_strength": uniform(0.0, 2.0),
    },
    "adaboost": {
        "learning_rate": uniform(0.01, 1.4),
    },
    "lightgbm": {
        "learning_rate": uniform(0.01, 0.29),
        "num_leaves": randint(16, 140),
        "feature_fraction": uniform(0.55, 0.45),
        "bagging_fraction": uniform(0.55, 0.45),
        "min_child_samples": randint(5, 80),
        "reg_lambda": loguniform(1e-3, 30),
        "min_split_gain": uniform(0.0, 1.5),
    },
}

BALANCE_STRATEGIES = [
    "base",
    "smote",
    "adasyn",
    "classweights",
    "smote_classweights",
    "adasyn_classweights",
]


def model_is_available(name):
    if name == "xgboost":
        return xgb_available
    if name == "catboost":
        return cat_available
    if name == "lightgbm":
        return lgb_available
    return True


AVAILABLE_MODELS = [m for m in MODEL_SPACES if model_is_available(m)]
print("Models to train:", AVAILABLE_MODELS)
print("Balancing strategies:", BALANCE_STRATEGIES)

Models to train: ['log_reg', 'knn', 'naive_bayes', 'random_forest', 'xgboost', 'catboost', 'adaboost', 'lightgbm']
Balancing strategies: ['base', 'smote', 'adasyn', 'classweights', 'smote_classweights', 'adasyn_classweights']


In [37]:
def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        if p.ndim == 2:
            return np.clip(p[:, 1], 1e-6, 1 - 1e-6)
        return np.clip(p.reshape(-1), 1e-6, 1 - 1e-6)
    if hasattr(model, "decision_function"):
        return np.clip(expit(model.decision_function(X)), 1e-6, 1 - 1e-6)
    return np.clip(model.predict(X).astype(float), 1e-6, 1 - 1e-6)


def metric_pack(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.clip(np.asarray(y_prob), 1e-6, 1 - 1e-6)
    y_pred = (y_prob >= threshold).astype(int)

    if np.unique(y_true).size < 2:
        auc_val = 0.5
    else:
        auc_val = roc_auc_score(y_true, y_prob)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": auc_val,
        "logloss": log_loss(y_true, y_prob, labels=[0, 1]),
    }


def stage_score(metrics):
    auc = metrics.get("auc_mean", metrics.get("auc", 0.0))
    logloss = metrics.get("logloss_mean", metrics.get("logloss", 10.0))
    f1 = metrics.get("f1_mean", metrics.get("f1", 0.0))
    auc_std = metrics.get("auc_std", 0.0)
    logloss_std = metrics.get("logloss_std", 0.0)
    return (auc - 0.70 * logloss + 0.15 * f1) - (0.05 * auc_std + 0.02 * logloss_std)


CLASSWEIGHT_STRATEGIES = {"classweights", "smote_classweights", "adasyn_classweights"}
RESAMPLING_STRATEGIES = {"smote", "adasyn", "smote_classweights", "adasyn_classweights"}


def get_balance_strategy(params):
    return str(params.get("__balance_strategy", "base"))


def strip_internal_params(params):
    return {k: v for k, v in params.items() if not str(k).startswith("__")}


def model_supports_intrinsic_class_weight(model_name):
    return model_name in {"log_reg", "random_forest", "xgboost", "lightgbm", "catboost"}


def model_supports_sample_weight(model_name):
    return model_name in {"log_reg", "naive_bayes", "random_forest", "xgboost", "catboost", "adaboost", "lightgbm"}


def positive_class_scale(y):
    y_arr = np.asarray(y).astype(int)
    pos = int((y_arr == 1).sum())
    neg = int((y_arr == 0).sum())
    if pos <= 0:
        return 1.0
    return float(max(1.0, neg / max(pos, 1)))


def compute_balanced_sample_weight(y):
    y_arr = np.asarray(y).astype(int)
    n = len(y_arr)
    pos = int((y_arr == 1).sum())
    neg = int((y_arr == 0).sum())
    if pos == 0 or neg == 0:
        return np.ones(n, dtype=float)
    w_pos = n / (2.0 * pos)
    w_neg = n / (2.0 * neg)
    return np.where(y_arr == 1, w_pos, w_neg).astype(float)


def oversample_minority_to_balance(X_in, y_in):
    X_work = X_in.reset_index(drop=True)
    y_work = y_in.reset_index(drop=True)

    y_arr = np.asarray(y_work).astype(int)
    idx_pos = np.where(y_arr == 1)[0]
    idx_neg = np.where(y_arr == 0)[0]

    if len(idx_pos) == 0 or len(idx_neg) == 0:
        return X_work, y_work

    if len(idx_pos) < len(idx_neg):
        minority_idx = idx_pos
        n_to_add = len(idx_neg) - len(idx_pos)
    else:
        minority_idx = idx_neg
        n_to_add = len(idx_pos) - len(idx_neg)

    if n_to_add <= 0:
        return X_work, y_work

    add_idx = np.random.choice(minority_idx, size=n_to_add, replace=True)
    X_add = X_work.iloc[add_idx]
    y_add = y_work.iloc[add_idx]

    X_bal = pd.concat([X_work, X_add], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y_work, y_add], axis=0).reset_index(drop=True)
    return X_bal, y_bal


def apply_resampling(X_in, y_in, strategy):
    X_work = X_in.reset_index(drop=True)
    y_work = y_in.reset_index(drop=True)

    if strategy not in RESAMPLING_STRATEGIES:
        return X_work, y_work

    sampler = None
    if strategy.startswith("smote") and smote_available:
        sampler = SMOTE(random_state=RANDOM_SEED)
    if strategy.startswith("adasyn") and adasyn_available:
        sampler = ADASYN(random_state=RANDOM_SEED)

    if sampler is None:
        return X_work, y_work

    try:
        X_res, y_res = sampler.fit_resample(X_work, y_work)
    except Exception:
        if strategy.startswith("adasyn") and smote_available:
            X_res, y_res = SMOTE(random_state=RANDOM_SEED).fit_resample(X_work, y_work)
        else:
            return X_work, y_work

    if not isinstance(X_res, pd.DataFrame):
        X_res = pd.DataFrame(X_res, columns=X_work.columns)
    if not isinstance(y_res, pd.Series):
        y_res = pd.Series(y_res, name=y_work.name)

    return X_res.reset_index(drop=True), y_res.reset_index(drop=True)


def prepare_training_data(model_name, Xtr, ytr, balance_strategy):
    strategy = balance_strategy or "base"
    X_bal, y_bal = apply_resampling(Xtr, ytr, strategy)

    sample_weight = None
    if strategy in CLASSWEIGHT_STRATEGIES:
        if model_supports_intrinsic_class_weight(model_name):
            sample_weight = None
        elif model_supports_sample_weight(model_name):
            sample_weight = compute_balanced_sample_weight(y_bal)
        else:
            X_bal, y_bal = oversample_minority_to_balance(X_bal, y_bal)

    return X_bal, y_bal, sample_weight


def build_model(model_name, params, epoch_budget, balance_strategy="base", y_train_ref=None):
    p = deepcopy(strip_internal_params(params))
    use_class_weight = balance_strategy in CLASSWEIGHT_STRATEGIES
    class_scale = positive_class_scale(y_train_ref) if y_train_ref is not None else 1.0
    use_gpu = bool(USE_GPU_WHEN_AVAILABLE and torch_cuda_available)

    if model_name == "log_reg":
        solver = p["solver"]
        penalty = p["penalty"]
        if penalty == "l1" and solver not in {"liblinear", "saga"}:
            solver = "liblinear"
        return LogisticRegression(
            C=float(p["C"]),
            penalty=penalty,
            solver=solver,
            class_weight="balanced" if use_class_weight else None,
            max_iter=max(500, int(epoch_budget * 15)),
            random_state=RANDOM_SEED,
        )

    if model_name == "knn":
        return KNeighborsClassifier(
            n_neighbors=int(p["n_neighbors"]),
            weights=p["weights"],
            p=int(p["p"]),
        )

    if model_name == "naive_bayes":
        return GaussianNB(var_smoothing=float(p["var_smoothing"]))

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=int(epoch_budget),
            max_depth=p["max_depth"],
            min_samples_split=int(p["min_samples_split"]),
            min_samples_leaf=int(p["min_samples_leaf"]),
            max_features=p["max_features"],
            class_weight="balanced_subsample" if use_class_weight else None,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )

    if model_name == "xgboost":
        xgb_kwargs = {
            "n_estimators": int(epoch_budget),
            "learning_rate": float(p["learning_rate"]),
            "max_depth": int(p["max_depth"]),
            "subsample": float(p["subsample"]),
            "colsample_bytree": float(p["colsample_bytree"]),
            "min_child_weight": float(p.get("min_child_weight", 1.0)),
            "gamma": float(p.get("gamma", 0.0)),
            "reg_lambda": float(p["reg_lambda"]),
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
            "verbosity": 0,
            "tree_method": "gpu_hist" if use_gpu else "hist",
        }
        if use_class_weight:
            xgb_kwargs["scale_pos_weight"] = float(class_scale)
        return XGBClassifier(**xgb_kwargs)

    if model_name == "catboost":
        cat_kwargs = {
            "iterations": int(epoch_budget),
            "learning_rate": float(p["learning_rate"]),
            "depth": int(p["depth"]),
            "l2_leaf_reg": float(p["l2_leaf_reg"]),
            "random_strength": float(p.get("random_strength", 1.0)),
            "loss_function": "Logloss",
            "random_seed": RANDOM_SEED,
            "verbose": False,
        }
        if use_class_weight:
            cat_kwargs["class_weights"] = [1.0, float(class_scale)]
        if use_gpu:
            cat_kwargs["task_type"] = "GPU"
            cat_kwargs["devices"] = "0"
        return CatBoostClassifier(**cat_kwargs)

    if model_name == "adaboost":
        return AdaBoostClassifier(
            n_estimators=int(epoch_budget),
            learning_rate=float(p["learning_rate"]),
            random_state=RANDOM_SEED,
        )

    if model_name == "lightgbm":
        lgb_kwargs = {
            "n_estimators": int(epoch_budget),
            "learning_rate": float(p["learning_rate"]),
            "num_leaves": int(p["num_leaves"]),
            "feature_fraction": float(p["feature_fraction"]),
            "bagging_fraction": float(p["bagging_fraction"]),
            "min_child_samples": int(p["min_child_samples"]),
            "reg_lambda": float(p.get("reg_lambda", 0.0)),
            "min_split_gain": float(p.get("min_split_gain", 0.0)),
            "objective": "binary",
            "random_state": RANDOM_SEED,
            "verbosity": -1,
            "n_jobs": -1,
        }
        if use_class_weight:
            lgb_kwargs["class_weight"] = "balanced"
        if use_gpu:
            lgb_kwargs["device_type"] = "gpu"
        return LGBMClassifier(**lgb_kwargs)

    raise ValueError(f"Unsupported model: {model_name}")


def _fit_with_optional_sample_weight(model, Xtr, ytr, sample_weight=None, **fit_kwargs):
    kwargs = dict(fit_kwargs)
    if sample_weight is not None:
        kwargs["sample_weight"] = sample_weight
    try:
        model.fit(Xtr, ytr, **kwargs)
    except TypeError:
        kwargs.pop("sample_weight", None)
        model.fit(Xtr, ytr, **kwargs)
    return model


def _fallback_model_to_cpu(model):
    model_name = type(model).__name__.lower()
    try:
        if "xgb" in model_name:
            model.set_params(tree_method="hist")
        elif "catboost" in model_name:
            model.set_params(task_type="CPU")
        elif "lgbm" in model_name:
            model.set_params(device_type="cpu")
    except Exception:
        pass
    return model


def fit_model(model, Xtr, ytr, Xval=None, yval=None, early_stopping_rounds=None, sample_weight=None):
    model_name = type(model).__name__.lower()

    def _fit_once(model_obj):
        if "catboost" in model_name and Xval is not None:
            return _fit_with_optional_sample_weight(
                model_obj,
                Xtr,
                ytr,
                sample_weight=sample_weight,
                eval_set=(Xval, yval),
                early_stopping_rounds=early_stopping_rounds,
                verbose=False,
            )

        if "xgb" in model_name and Xval is not None:
            return _fit_with_optional_sample_weight(
                model_obj,
                Xtr,
                ytr,
                sample_weight=sample_weight,
                eval_set=[(Xval, yval)],
                verbose=False,
            )

        if "lgbm" in model_name and Xval is not None and lgb is not None:
            callbacks = []
            if early_stopping_rounds is not None:
                callbacks = [lgb.early_stopping(early_stopping_rounds, verbose=False)]
            return _fit_with_optional_sample_weight(
                model_obj,
                Xtr,
                ytr,
                sample_weight=sample_weight,
                eval_set=[(Xval, yval)],
                eval_metric="binary_logloss",
                callbacks=callbacks,
            )

        return _fit_with_optional_sample_weight(model_obj, Xtr, ytr, sample_weight=sample_weight)

    try:
        return _fit_once(model)
    except Exception as e:
        msg = str(e).lower()
        if USE_GPU_WHEN_AVAILABLE and any(tok in msg for tok in ["gpu", "cuda", "device"]):
            model = _fallback_model_to_cpu(model)
            return _fit_once(model)
        raise


def evaluate_params_cv(model_name, params, X_data, y_data, epoch_budget, n_splits=5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    y_array = np.asarray(y_data).astype(int)
    rows = []

    strategy = get_balance_strategy(params)

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X_data, y_array), start=1):
        Xtr = X_data.iloc[tr_idx]
        Xva = X_data.iloc[va_idx]
        ytr = y_data.iloc[tr_idx]
        yva = y_data.iloc[va_idx]

        Xtr_fit, ytr_fit, sample_weight = prepare_training_data(model_name, Xtr, ytr, strategy)
        model = build_model(
            model_name,
            params,
            epoch_budget=epoch_budget,
            balance_strategy=strategy,
            y_train_ref=ytr_fit,
        )
        model = fit_model(
            model,
            Xtr_fit,
            ytr_fit,
            Xva,
            yva,
            early_stopping_rounds=10 if epoch_budget >= STAGE2_EPOCHS else None,
            sample_weight=sample_weight,
        )
        p_val = safe_predict_proba(model, Xva)
        met = metric_pack(yva, p_val, threshold=0.5)
        met["fold"] = fold
        rows.append(met)

    fold_df = pd.DataFrame(rows)
    summary = {
        "auc_mean": float(fold_df["auc"].mean()),
        "auc_std": float(fold_df["auc"].std(ddof=0)),
        "logloss_mean": float(fold_df["logloss"].mean()),
        "logloss_std": float(fold_df["logloss"].std(ddof=0)),
        "f1_mean": float(fold_df["f1"].mean()),
        "f1_std": float(fold_df["f1"].std(ddof=0)),
        "balance_strategy": strategy,
    }
    summary["stage_score"] = stage_score(summary)
    return summary


def sample_stage1_params(model_name, n_trials):
    n_per_strategy = max(1, int(np.ceil(n_trials / max(1, len(BALANCE_STRATEGIES)))))
    rows = []
    for idx, strategy in enumerate(BALANCE_STRATEGIES):
        sampled = list(
            ParameterSampler(
                MODEL_SPACES[model_name],
                n_iter=n_per_strategy,
                random_state=RANDOM_SEED + (idx + 1) * 101,
            )
        )
        for p in sampled:
            p2 = deepcopy(p)
            p2["__balance_strategy"] = strategy
            rows.append(p2)
    return dedupe_param_dicts(rows)


def dedupe_param_dicts(params_list):
    uniq = []
    seen = set()
    for p in params_list:
        if not isinstance(p, dict):
            continue
        key = json.dumps(p, sort_keys=True, default=str)
        if key not in seen:
            seen.add(key)
            uniq.append(p)
    return uniq

In [ ]:
# Rigorous optimization profile override for training/search phases.
# Run this before Stage 1 and Stage 2 search cells.
RIGOROUS_OPT_PROFILE = True
N_JOBS = -1

STAGE1_EPOCHS = max(STAGE1_EPOCHS, 60)
STAGE2_EPOCHS = max(STAGE2_EPOCHS, 180)
FINAL_EPOCHS = max(FINAL_EPOCHS, 700)

STAGE1_TRIALS_PER_MODEL = max(STAGE1_TRIALS_PER_MODEL, 260)
STAGE2_REFINEMENTS_PER_TOP_CONFIG = max(STAGE2_REFINEMENTS_PER_TOP_CONFIG, 42)
TOP_K_STAGE1 = max(TOP_K_STAGE1, 14)
TOP_K_STAGE2 = max(TOP_K_STAGE2, 6)

CV_FOLDS_STAGE1 = max(CV_FOLDS_STAGE1, 6)
CV_FOLDS_STAGE2 = max(CV_FOLDS_STAGE2, 8)
STACK_OOF_FOLDS = max(STACK_OOF_FOLDS, 8)
STACK_META_TRIALS = max(STACK_META_TRIALS, 800)
STACK_BLEND_TRIALS = max(STACK_BLEND_TRIALS, 12000)

RIGOROUS_REPEATS_STAGE1 = 2
RIGOROUS_REPEATS_STAGE2 = 2
RIGOROUS_THRESHOLD_GRID = np.round(np.arange(0.35, 0.70, 0.05), 2)

# Force n_jobs = -1 for estimators that support it.
_ORIG_LOGISTIC_REGRESSION = LogisticRegression
def LogisticRegression(*args, **kwargs):
    kwargs.setdefault("n_jobs", N_JOBS)
    return _ORIG_LOGISTIC_REGRESSION(*args, **kwargs)

_ORIG_KNEIGHBORS_CLASSIFIER = KNeighborsClassifier
def KNeighborsClassifier(*args, **kwargs):
    kwargs.setdefault("n_jobs", N_JOBS)
    return _ORIG_KNEIGHBORS_CLASSIFIER(*args, **kwargs)

_ORIG_BUILD_MODEL = build_model
def build_model(model_name, params, epoch_budget, balance_strategy="base", y_train_ref=None):
    model = _ORIG_BUILD_MODEL(model_name, params, epoch_budget, balance_strategy, y_train_ref)
    try:
        cls_name = type(model).__name__.lower()
        if "catboost" in cls_name:
            model.set_params(thread_count=N_JOBS)
        elif any(tok in cls_name for tok in ["randomforest", "xgb", "lgbm"]):
            model.set_params(n_jobs=N_JOBS)
    except Exception:
        pass
    return model


def _fold_objective(m):
    # Favor discrimination and minority sensitivity while penalizing loss.
    return (0.50 * m["auc"] + 0.30 * m["f1"] + 0.20 * m["recall"]) - (0.55 * m["logloss"])


def stage_score(metrics):
    auc = float(metrics.get("auc_mean", metrics.get("auc", 0.0)))
    logloss = float(metrics.get("logloss_mean", metrics.get("logloss", 10.0)))
    f1 = float(metrics.get("f1_mean", metrics.get("f1", 0.0)))
    precision = float(metrics.get("precision_mean", metrics.get("precision", 0.0)))
    recall = float(metrics.get("recall_mean", metrics.get("recall", 0.0)))
    auc_std = float(metrics.get("auc_std", 0.0))
    logloss_std = float(metrics.get("logloss_std", 0.0))
    f1_std = float(metrics.get("f1_std", 0.0))

    return (0.45 * auc + 0.25 * f1 + 0.20 * recall + 0.10 * precision - 0.55 * logloss) - (
        0.06 * auc_std + 0.03 * logloss_std + 0.02 * f1_std
    )


def evaluate_params_cv(model_name, params, X_data, y_data, epoch_budget, n_splits=5):
    strategy = get_balance_strategy(params)
    y_array = np.asarray(y_data).astype(int)
    repeats = RIGOROUS_REPEATS_STAGE2 if epoch_budget >= STAGE2_EPOCHS else RIGOROUS_REPEATS_STAGE1

    rows = []
    for repeat_idx in range(repeats):
        splitter = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_SEED + (repeat_idx * 1009) + int(epoch_budget),
        )

        for fold, (tr_idx, va_idx) in enumerate(splitter.split(X_data, y_array), start=1):
            Xtr = X_data.iloc[tr_idx]
            Xva = X_data.iloc[va_idx]
            ytr = y_data.iloc[tr_idx]
            yva = y_data.iloc[va_idx]

            Xtr_fit, ytr_fit, sample_weight = prepare_training_data(model_name, Xtr, ytr, strategy)
            model = build_model(
                model_name,
                params,
                epoch_budget=epoch_budget,
                balance_strategy=strategy,
                y_train_ref=ytr_fit,
            )
            model = fit_model(
                model,
                Xtr_fit,
                ytr_fit,
                Xva,
                yva,
                early_stopping_rounds=10 if epoch_budget >= STAGE2_EPOCHS else None,
                sample_weight=sample_weight,
            )

            p_val = safe_predict_proba(model, Xva)
            best_fold_met = None
            best_fold_obj = -np.inf
            best_threshold = 0.5

            for thr in RIGOROUS_THRESHOLD_GRID:
                met = metric_pack(yva, p_val, threshold=float(thr))
                fold_obj = _fold_objective(met)
                if fold_obj > best_fold_obj:
                    best_fold_obj = fold_obj
                    best_fold_met = met
                    best_threshold = float(thr)

            best_fold_met["fold"] = int(fold)
            best_fold_met["repeat"] = int(repeat_idx + 1)
            best_fold_met["best_threshold"] = float(best_threshold)
            rows.append(best_fold_met)

    fold_df = pd.DataFrame(rows)
    summary = {
        "auc_mean": float(fold_df["auc"].mean()),
        "auc_std": float(fold_df["auc"].std(ddof=0)),
        "logloss_mean": float(fold_df["logloss"].mean()),
        "logloss_std": float(fold_df["logloss"].std(ddof=0)),
        "f1_mean": float(fold_df["f1"].mean()),
        "f1_std": float(fold_df["f1"].std(ddof=0)),
        "precision_mean": float(fold_df["precision"].mean()),
        "precision_std": float(fold_df["precision"].std(ddof=0)),
        "recall_mean": float(fold_df["recall"].mean()),
        "recall_std": float(fold_df["recall"].std(ddof=0)),
        "threshold_mean": float(fold_df["best_threshold"].mean()),
        "threshold_std": float(fold_df["best_threshold"].std(ddof=0)),
        "balance_strategy": strategy,
        "cv_repeats": int(repeats),
    }
    summary["stage_score"] = stage_score(summary)
    return summary


print("Rigorous optimization profile enabled:")
print(
    {
        "n_jobs": N_JOBS,
        "stage1_trials_per_model": STAGE1_TRIALS_PER_MODEL,
        "stage2_refinements_per_top_config": STAGE2_REFINEMENTS_PER_TOP_CONFIG,
        "top_k": {"stage1": TOP_K_STAGE1, "stage2": TOP_K_STAGE2},
        "cv_folds": {"stage1": CV_FOLDS_STAGE1, "stage2": CV_FOLDS_STAGE2},
        "cv_repeats": {"stage1": RIGOROUS_REPEATS_STAGE1, "stage2": RIGOROUS_REPEATS_STAGE2},
        "epochs": {"stage1": STAGE1_EPOCHS, "stage2": STAGE2_EPOCHS, "final": FINAL_EPOCHS},
        "stack": {
            "oof_folds": STACK_OOF_FOLDS,
            "meta_trials": STACK_META_TRIALS,
            "blend_trials": STACK_BLEND_TRIALS,
        },
        "threshold_grid": RIGOROUS_THRESHOLD_GRID.tolist(),
    }
)

In [38]:
stage1_results = {}
stage1_top_params = {}

for model_name in AVAILABLE_MODELS:
    trials = sample_stage1_params(model_name, STAGE1_TRIALS_PER_MODEL)
    rows = []
    print(
        f"Stage 1 CV search -> {model_name} "
        f"(epochs={STAGE1_EPOCHS}, folds={CV_FOLDS_STAGE1}, candidates={len(trials)})"
    )

    for i, params in enumerate(trials, start=1):
        strategy = get_balance_strategy(params)
        try:
            cv_met = evaluate_params_cv(
                model_name,
                params,
                X_train_final,
                y_train,
                epoch_budget=STAGE1_EPOCHS,
                n_splits=CV_FOLDS_STAGE1,
            )
            rows.append(
                {
                    "trial": i,
                    "balance_strategy": strategy,
                    "params": params,
                    "auc_mean": cv_met["auc_mean"],
                    "auc_std": cv_met["auc_std"],
                    "logloss_mean": cv_met["logloss_mean"],
                    "logloss_std": cv_met["logloss_std"],
                    "f1_mean": cv_met["f1_mean"],
                    "f1_std": cv_met["f1_std"],
                    "stage_score": cv_met["stage_score"],
                }
            )
        except Exception as e:
            rows.append(
                {
                    "trial": i,
                    "balance_strategy": strategy,
                    "params": params,
                    "error": str(e),
                    "stage_score": -999.0,
                }
            )

        if i % 10 == 0 or i == len(trials):
            print(f"  completed {i}/{len(trials)}")

    df_stage = pd.DataFrame(rows).sort_values(["stage_score", "auc_mean"], ascending=False).reset_index(drop=True)
    stage1_results[model_name] = df_stage
    stage1_top_params[model_name] = df_stage.head(TOP_K_STAGE1)["params"].tolist()

stage1_summary_rows = []
for m in AVAILABLE_MODELS:
    df_m = stage1_results[m]
    if len(df_m) == 0:
        stage1_summary_rows.append(
            {
                "model": m,
                "best_stage1_auc_mean": np.nan,
                "best_stage1_logloss_mean": np.nan,
                "best_stage1_f1_mean": np.nan,
                "best_balance_strategy": None,
                "trials": 0,
            }
        )
        continue

    top = df_m.iloc[0]
    stage1_summary_rows.append(
        {
            "model": m,
            "best_stage1_auc_mean": top.get("auc_mean", np.nan),
            "best_stage1_logloss_mean": top.get("logloss_mean", np.nan),
            "best_stage1_f1_mean": top.get("f1_mean", np.nan),
            "best_balance_strategy": top.get("balance_strategy", None),
            "trials": len(df_m),
        }
    )

stage1_summary = pd.DataFrame(stage1_summary_rows).sort_values("best_stage1_auc_mean", ascending=False)

stage1_summary

Stage 1 CV search -> log_reg (epochs=40, folds=5, candidates=144)
  completed 10/144
  completed 20/144
  completed 30/144
  completed 40/144
  completed 50/144
  completed 60/144
  completed 70/144
  completed 80/144
  completed 90/144
  completed 100/144
  completed 110/144
  completed 120/144
  completed 130/144
  completed 140/144
  completed 144/144
Stage 1 CV search -> knn (epochs=40, folds=5, candidates=136)
  completed 10/136
  completed 20/136
  completed 30/136
  completed 40/136
  completed 50/136
  completed 60/136
  completed 70/136
  completed 80/136
  completed 90/136
  completed 100/136
  completed 110/136
  completed 120/136
  completed 130/136
  completed 136/136
Stage 1 CV search -> naive_bayes (epochs=40, folds=5, candidates=144)
  completed 10/144
  completed 20/144
  completed 30/144
  completed 40/144
  completed 50/144
  completed 60/144
  completed 70/144
  completed 80/144
  completed 90/144
  completed 100/144
  completed 110/144
  completed 120/144
  complet

,model,best_stage1_auc_mean,best_stage1_logloss_mean,best_stage1_f1_mean,best_balance_strategy,trials
7,lightgbm,0.851172,0.443074,0.671541,base,144
4,xgboost,0.850971,0.443059,0.672557,base,144
5,catboost,0.850950,0.443939,0.673672,base,144
3,random_forest,0.848830,0.446267,0.669173,base,144
0,log_reg,0.845859,0.452908,0.652134,base,144
1,knn,0.841784,0.458813,0.653968,base,136
6,adaboost,0.839694,0.492838,0.664705,base,144
2,naive_bayes,0.815352,1.113381,0.672497,base,144


In [39]:
def refine_candidates(base_params_list, n_refinements_per_base=4):
    refined = []
    for base in base_params_list:
        if not isinstance(base, dict):
            continue

        refined.append(deepcopy(base))
        for _ in range(n_refinements_per_base):
            cand = {}
            for k, v in base.items():
                if str(k).startswith("__"):
                    cand[k] = v
                elif isinstance(v, (int, np.integer)):
                    factor = np.random.uniform(0.65, 1.35)
                    cand[k] = max(1, int(round(v * factor)))
                elif isinstance(v, (float, np.floating)):
                    factor = np.random.uniform(0.65, 1.35)
                    cand[k] = max(1e-6, float(v * factor))
                else:
                    cand[k] = v
            refined.append(cand)
    return dedupe_param_dicts(refined)


stage2_results = {}
best_configs = {}

for model_name in AVAILABLE_MODELS:
    local_refined = refine_candidates(
        stage1_top_params.get(model_name, []),
        n_refinements_per_base=STAGE2_REFINEMENTS_PER_TOP_CONFIG,
    )
    exploration_trials = sample_stage1_params(
        model_name,
        max(16, STAGE1_TRIALS_PER_MODEL // 3),
    )
    candidates = dedupe_param_dicts(local_refined + exploration_trials)

    rows = []
    print(
        f"Stage 2 CV search -> {model_name} "
        f"(epochs={STAGE2_EPOCHS}, folds={CV_FOLDS_STAGE2}, candidates={len(candidates)})"
    )

    for i, params in enumerate(candidates, start=1):
        strategy = get_balance_strategy(params)
        try:
            cv_met = evaluate_params_cv(
                model_name,
                params,
                X_train_final,
                y_train,
                epoch_budget=STAGE2_EPOCHS,
                n_splits=CV_FOLDS_STAGE2,
            )
            rows.append(
                {
                    "trial": i,
                    "balance_strategy": strategy,
                    "params": params,
                    "auc_mean": cv_met["auc_mean"],
                    "auc_std": cv_met["auc_std"],
                    "logloss_mean": cv_met["logloss_mean"],
                    "logloss_std": cv_met["logloss_std"],
                    "f1_mean": cv_met["f1_mean"],
                    "f1_std": cv_met["f1_std"],
                    "stage_score": cv_met["stage_score"],
                }
            )
        except Exception as e:
            rows.append(
                {
                    "trial": i,
                    "balance_strategy": strategy,
                    "params": params,
                    "error": str(e),
                    "stage_score": -999.0,
                }
            )

        if i % 10 == 0 or i == len(candidates):
            print(f"  completed {i}/{len(candidates)}")

    df_stage2 = pd.DataFrame(rows).sort_values(["stage_score", "auc_mean"], ascending=False).reset_index(drop=True)
    stage2_results[model_name] = df_stage2
    best_configs[model_name] = df_stage2.head(TOP_K_STAGE2)["params"].tolist()

stage2_summary_rows = []
for m in AVAILABLE_MODELS:
    df_m = stage2_results[m]
    if len(df_m) == 0:
        stage2_summary_rows.append(
            {
                "model": m,
                "best_stage2_auc_mean": np.nan,
                "best_stage2_logloss_mean": np.nan,
                "best_stage2_f1_mean": np.nan,
                "best_balance_strategy": None,
                "candidates": 0,
            }
        )
        continue

    top = df_m.iloc[0]
    stage2_summary_rows.append(
        {
            "model": m,
            "best_stage2_auc_mean": top.get("auc_mean", np.nan),
            "best_stage2_logloss_mean": top.get("logloss_mean", np.nan),
            "best_stage2_f1_mean": top.get("f1_mean", np.nan),
            "best_balance_strategy": top.get("balance_strategy", None),
            "candidates": len(df_m),
        }
    )

stage2_summary = pd.DataFrame(stage2_summary_rows).sort_values("best_stage2_auc_mean", ascending=False)

stage2_summary

Stage 2 CV search -> log_reg (epochs=120, folds=6, candidates=334)
  completed 10/334
  completed 20/334
  completed 30/334
  completed 40/334
  completed 50/334
  completed 60/334
  completed 70/334
  completed 80/334
  completed 90/334
  completed 100/334
  completed 110/334
  completed 120/334
  completed 130/334
  completed 140/334
  completed 150/334
  completed 160/334
  completed 170/334
  completed 180/334
  completed 190/334
  completed 200/334
  completed 210/334
  completed 220/334
  completed 230/334
  completed 240/334
  completed 250/334
  completed 260/334
  completed 270/334
  completed 280/334
  completed 290/334
  completed 300/334
  completed 310/334
  completed 320/334
  completed 330/334
  completed 334/334
Stage 2 CV search -> knn (epochs=120, folds=6, candidates=217)
  completed 10/217
  completed 20/217
  completed 30/217
  completed 40/217
  completed 50/217
  completed 60/217
  completed 70/217
  completed 80/217
  completed 90/217
  completed 100/217
  comple

,model,best_stage2_auc_mean,best_stage2_logloss_mean,best_stage2_f1_mean,best_balance_strategy,candidates
7,lightgbm,0.851944,0.442057,0.672660,base,334
5,catboost,0.851795,0.442742,0.673270,base,337
4,xgboost,0.851553,0.442471,0.671905,base,336
3,random_forest,0.849989,0.444723,0.671499,base,334
0,log_reg,0.845849,0.452912,0.652159,base,334
6,adaboost,0.844021,0.496308,0.660861,base,335
1,knn,0.842583,0.455855,0.651870,base,217
2,naive_bayes,0.815684,1.111245,0.673000,base,53


In [52]:
TOP_K_PER_BALANCE_STYLE = 2

def evaluate_candidate_cv_metrics(model_name, params):
    splitter = StratifiedKFold(n_splits=CV_FOLDS_STAGE2, shuffle=True, random_state=RANDOM_SEED)
    y_array = np.asarray(y_train).astype(int)
    strategy = get_balance_strategy(params)
    fold_rows = []

    for tr_idx, va_idx in splitter.split(X_train_final, y_array):
        Xtr = X_train_final.iloc[tr_idx]
        Xva = X_train_final.iloc[va_idx]
        ytr = y_train.iloc[tr_idx]
        yva = y_train.iloc[va_idx]

        Xtr_fit, ytr_fit, sample_weight = prepare_training_data(model_name, Xtr, ytr, strategy)
        model = build_model(
            model_name,
            params,
            epoch_budget=STAGE2_EPOCHS,
            balance_strategy=strategy,
            y_train_ref=ytr_fit,
        )
        model = fit_model(
            model,
            Xtr_fit,
            ytr_fit,
            Xva,
            yva,
            early_stopping_rounds=10,
            sample_weight=sample_weight,
        )
        p_val = safe_predict_proba(model, Xva)
        fold_rows.append(metric_pack(yva, p_val, threshold=0.5))

    fold_df = pd.DataFrame(fold_rows)
    return {
        "accuracy_mean": float(fold_df["accuracy"].mean()),
        "accuracy_std": float(fold_df["accuracy"].std(ddof=0)),
        "precision_mean": float(fold_df["precision"].mean()),
        "precision_std": float(fold_df["precision"].std(ddof=0)),
        "recall_mean": float(fold_df["recall"].mean()),
        "recall_std": float(fold_df["recall"].std(ddof=0)),
    }

best_configs = {}
best2_rows = []

for model_name in AVAILABLE_MODELS:
    df_stage2 = stage2_results.get(model_name, pd.DataFrame()).copy()
    if df_stage2.empty:
        best_configs[model_name] = []
        continue

    valid_df = df_stage2[df_stage2["params"].apply(lambda p: isinstance(p, dict))].copy()
    if valid_df.empty:
        best_configs[model_name] = []
        continue

    top2 = (
        valid_df.sort_values(
            ["balance_strategy", "stage_score", "auc_mean"],
            ascending=[True, False, False],
        )
        .groupby("balance_strategy", as_index=False, group_keys=False)
        .head(TOP_K_PER_BALANCE_STYLE)
        .reset_index(drop=True)
    )
    top2["rank_within_balance_strategy"] = top2.groupby("balance_strategy").cumcount() + 1
    top2["model"] = model_name

    best_configs[model_name] = top2["params"].tolist()
    best2_rows.append(top2)

if best2_rows:
    stage2_best2_df = pd.concat(best2_rows, ignore_index=True)
else:
    stage2_best2_df = pd.DataFrame(
        columns=[
            "model",
            "balance_strategy",
            "rank_within_balance_strategy",
            "trial",
            "params",
            "auc_mean",
            "auc_std",
            "logloss_mean",
            "logloss_std",
            "f1_mean",
            "f1_std",
            "accuracy_mean",
            "accuracy_std",
            "precision_mean",
            "precision_std",
            "recall_mean",
            "recall_std",
            "stage_score",
            "error",
            "metric_error",
        ]
    )

metric_rows = []
total_rows = len(stage2_best2_df)
for i, row in stage2_best2_df.reset_index(drop=True).iterrows():
    try:
        metrics = evaluate_candidate_cv_metrics(row["model"], row["params"])
        metrics["metric_error"] = None
    except Exception as e:
        metrics = {
            "accuracy_mean": np.nan,
            "accuracy_std": np.nan,
            "precision_mean": np.nan,
            "precision_std": np.nan,
            "recall_mean": np.nan,
            "recall_std": np.nan,
            "metric_error": str(e),
        }
    metric_rows.append(metrics)

    if (i + 1) % 10 == 0 or (i + 1) == total_rows:
        print(f"  computed CV metrics {i + 1}/{total_rows}")

if metric_rows:
    metric_df = pd.DataFrame(metric_rows)
    stage2_best2_df = pd.concat([stage2_best2_df.reset_index(drop=True), metric_df], axis=1)

stage2_best2_path = ARTIFACT_DIR / "stage2_best2_per_balance_per_model_2015_rebuild.csv"
try:
    stage2_best2_df.to_csv(stage2_best2_path, index=False)
except PermissionError:
    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    stage2_best2_path = ARTIFACT_DIR / f"stage2_best2_per_balance_per_model_2015_rebuild_{ts}.csv"
    stage2_best2_df.to_csv(stage2_best2_path, index=False)

print(f"Saved: {stage2_best2_path}")
print(f"Exists after save: {stage2_best2_path.exists()}")
print(f"Rows: {len(stage2_best2_df)}")
print(f"Strategies present: {sorted(stage2_best2_df['balance_strategy'].dropna().unique().tolist()) if len(stage2_best2_df) > 0 else []}")
print("Candidates kept per model/strategy:")
print(
    stage2_best2_df.groupby(["model", "balance_strategy"]).size()
    .rename("kept")
    .to_string()
    if len(stage2_best2_df) > 0
    else "(none)"
 )

  computed CV metrics 10/96
  computed CV metrics 20/96
  computed CV metrics 30/96
  computed CV metrics 40/96
  computed CV metrics 50/96
  computed CV metrics 60/96
  computed CV metrics 70/96
  computed CV metrics 80/96
  computed CV metrics 90/96
  computed CV metrics 96/96
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\stage2_best2_per_balance_per_model_2015_rebuild.csv
Exists after save: True
Rows: 96
Strategies present: ['adasyn', 'adasyn_classweights', 'base', 'classweights', 'smote', 'smote_classweights']
Candidates kept per model/strategy:
model          balance_strategy   
adaboost       adasyn                 2
               adasyn_classweights    2
               base                   2
               classweights           2
               smote                  2
               smote_classweights     2
catboost       adasyn                 2
               adasyn_classweights    2
               base                   2
               classweig

In [51]:
# Save stage2_best2_df without recomputing metrics if the main file is locked.
stage2_best2_path = ARTIFACT_DIR / "stage2_best2_per_balance_per_model_2015_rebuild.csv"
try:
    stage2_best2_df.to_csv(stage2_best2_path, index=False)
except PermissionError:
    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    stage2_best2_path = ARTIFACT_DIR / f"stage2_best2_per_balance_per_model_2015_rebuild_{ts}.csv"
    stage2_best2_df.to_csv(stage2_best2_path, index=False)

print(f"Saved: {stage2_best2_path}")
print(
    stage2_best2_df[["model", "balance_strategy", "accuracy_mean", "precision_mean", "recall_mean"]]
    .head(12)
    .to_string(index=False)
 )

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\stage2_best2_per_balance_per_model_2015_rebuild_20260414_000824.csv
  model    balance_strategy  accuracy_mean  precision_mean  recall_mean
log_reg              adasyn       0.752050        0.592316     0.842089
log_reg              adasyn       0.752032        0.592289     0.842117
log_reg adasyn_classweights       0.749839        0.588402     0.851674
log_reg adasyn_classweights       0.749792        0.588357     0.851562
log_reg                base       0.772205        0.670052     0.635214
log_reg                base       0.772186        0.670013     0.635214
log_reg        classweights       0.759118        0.604465     0.820305
log_reg        classweights       0.759118        0.604538     0.819800
log_reg               smote       0.759647        0.605715     0.816792
log_reg               smote       0.759742        0.605871     0.816567
log_reg  smote_classweights       0.759676        0.605834     0.8162

In [40]:
final_models = {}
selected_final_params = {}
final_rows = []
final_selection_rows = []

for model_name in AVAILABLE_MODELS:
    top_list = best_configs.get(model_name, [])
    if not top_list:
        continue

    print(f"Final selection -> {model_name} (candidates={len(top_list)}, epochs={FINAL_EPOCHS})")

    best_model_obj = None
    best_params = None
    best_strategy = None
    best_valid_score = -np.inf
    best_valid_met = None

    for i, params in enumerate(top_list, start=1):
        strategy = get_balance_strategy(params)
        try:
            X_train_fit, y_train_fit, sample_weight = prepare_training_data(
                model_name,
                X_train_final,
                y_train,
                strategy,
            )

            model = build_model(
                model_name,
                params,
                epoch_budget=FINAL_EPOCHS,
                balance_strategy=strategy,
                y_train_ref=y_train_fit,
            )
            model = fit_model(
                model,
                X_train_fit,
                y_train_fit,
                X_valid_final,
                y_valid,
                early_stopping_rounds=15,
                sample_weight=sample_weight,
            )

            p_valid = safe_predict_proba(model, X_valid_final)
            valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
            valid_score = stage_score(valid_met)

            final_selection_rows.append(
                {
                    "model": model_name,
                    "balance_strategy": strategy,
                    "candidate_rank": i,
                    "params": params,
                    "valid_auc": valid_met["auc"],
                    "valid_logloss": valid_met["logloss"],
                    "valid_f1": valid_met["f1"],
                    "valid_stage_score": valid_score,
                }
            )

            if valid_score > best_valid_score:
                best_valid_score = valid_score
                best_model_obj = model
                best_params = params
                best_strategy = strategy
                best_valid_met = valid_met
        except Exception as e:
            final_selection_rows.append(
                {
                    "model": model_name,
                    "balance_strategy": strategy,
                    "candidate_rank": i,
                    "params": params,
                    "error": str(e),
                    "valid_stage_score": -999.0,
                }
            )

    if best_model_obj is None:
        print(f"Final train skipped for {model_name}: all candidates failed")
        continue

    final_models[model_name] = best_model_obj
    selected_final_params[model_name] = best_params
    joblib.dump(best_model_obj, MODEL_DIR / f"{model_name}.joblib")

    p_test = safe_predict_proba(best_model_obj, X_test_final)
    test_met = metric_pack(y_test, p_test, threshold=0.5)

    final_rows.append(
        {
            "model": model_name,
            "balance_strategy": best_strategy,
            "valid_auc": best_valid_met["auc"],
            "valid_logloss": best_valid_met["logloss"],
            "valid_f1": best_valid_met["f1"],
            "valid_stage_score": best_valid_score,
            "test_auc": test_met["auc"],
            "test_logloss": test_met["logloss"],
            "test_f1": test_met["f1"],
            "params": best_params,
        }
    )

final_results_df = pd.DataFrame(final_rows).sort_values(["valid_stage_score", "valid_auc"], ascending=False).reset_index(drop=True)
final_selection_df = pd.DataFrame(final_selection_rows).sort_values(["model", "valid_stage_score"], ascending=[True, False]).reset_index(drop=True)
final_selection_path = ARTIFACT_DIR / "final_candidate_selection_2015_rebuild.csv"
final_selection_df.to_csv(final_selection_path, index=False)

print(f"Saved final candidate comparisons: {final_selection_path}")
final_results_df

Final selection -> log_reg (candidates=4, epochs=500)
Final selection -> knn (candidates=4, epochs=500)
Final selection -> naive_bayes (candidates=4, epochs=500)
Final selection -> random_forest (candidates=4, epochs=500)
Final selection -> xgboost (candidates=4, epochs=500)
Final selection -> catboost (candidates=4, epochs=500)
Final selection -> adaboost (candidates=4, epochs=500)
Final selection -> lightgbm (candidates=4, epochs=500)
Saved final candidate comparisons: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\final_candidate_selection_2015_rebuild.csv


,model,balance_strategy,valid_auc,valid_logloss,valid_f1,valid_stage_score,test_auc,test_logloss,test_f1,params
0,lightgbm,base,0.857379,0.435462,0.679777,0.654522,0.845640,0.450082,0.665245,"{'bagging_fraction': 0.6119120988195819, 'feat..."
1,catboost,base,0.856878,0.436280,0.681159,0.653655,0.845556,0.450667,0.665337,"{'depth': 6, 'l2_leaf_reg': 8.391591857858911,..."
2,xgboost,base,0.855466,0.437278,0.673985,0.650469,0.844210,0.451994,0.662701,"{'colsample_bytree': 0.7163700117138441, 'gamm..."
3,random_forest,base,0.855233,0.438592,0.679806,0.650190,0.843493,0.452565,0.660188,"{'max_depth': 9, 'max_features': 0.53880911412..."
4,log_reg,base,0.849384,0.448666,0.661382,0.634525,0.839518,0.460678,0.643059,"{'C': 0.009499595830678125, 'penalty': 'l1', '..."
5,knn,base,0.846824,0.449786,0.656645,0.630471,0.835038,0.465889,0.643033,"{'n_neighbors': 136, 'p': 1, 'weights': 'dista..."
6,adaboost,base,0.850950,0.514680,0.670115,0.591191,0.840780,0.519521,0.653954,"{'learning_rate': 0.16465721447150078, '__bala..."
7,naive_bayes,base,0.819176,1.087332,0.678919,0.159881,0.808939,1.156953,0.667008,"{'var_smoothing': 1e-06, '__balance_strategy':..."


In [41]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true)
    y_prob = np.clip(np.asarray(y_prob), 1e-6, 1 - 1e-6)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.digitize(y_prob, bins) - 1

    ece = 0.0
    for i in range(n_bins):
        mask = bin_idx == i
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def platt_calibration(p_fit, y_fit, p_eval):
    lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
    lr.fit(p_fit.reshape(-1, 1), y_fit)
    return np.clip(lr.predict_proba(p_eval.reshape(-1, 1))[:, 1], 1e-6, 1 - 1e-6)


def isotonic_calibration(p_fit, y_fit, p_eval):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_fit, y_fit)
    return np.clip(iso.predict(p_eval), 1e-6, 1 - 1e-6)


def venn_abers_calibration(p_fit, y_fit, p_eval):
    if not venn_available:
        raise RuntimeError("venn-abers is not installed.")
    va = VennAbers()
    va.fit(np.column_stack([1.0 - p_fit, p_fit]), np.asarray(y_fit))
    _, p1 = va.predict_proba(np.column_stack([1.0 - p_eval, p_eval]))
    p1 = np.asarray(p1)
    if p1.ndim == 2:
        return np.clip(p1[:, 1], 1e-6, 1 - 1e-6)
    return np.clip(p1.reshape(-1), 1e-6, 1 - 1e-6)


def apply_calibration(method, p_fit, y_fit, p_eval):
    if method == "base":
        return np.clip(np.asarray(p_eval), 1e-6, 1 - 1e-6)
    if method == "platt":
        return platt_calibration(p_fit, y_fit, p_eval)
    if method == "isotonic":
        return isotonic_calibration(p_fit, y_fit, p_eval)
    if method == "venn_abers":
        return venn_abers_calibration(p_fit, y_fit, p_eval)
    raise ValueError(f"Unknown calibration method: {method}")


calibration_rows = []
calibrated_test_probs = {}

for model_name, model in final_models.items():
    p_valid_base = safe_predict_proba(model, X_valid_final)
    p_test_base = safe_predict_proba(model, X_test_final)

    methods = ["base", "platt", "isotonic"]
    if venn_available:
        methods.append("venn_abers")

    p_fit, p_eval, y_fit, y_eval = train_test_split(
        p_valid_base,
        y_valid.values,
        test_size=0.50,
        random_state=RANDOM_SEED,
        stratify=y_valid.values,
    )

    for method in methods:
        try:
            # Selection metrics are computed on validation holdout only.
            p_eval_cal = apply_calibration(method, p_fit, y_fit, p_eval)
            eval_m = metric_pack(y_eval, p_eval_cal, threshold=0.5)

            # Final test probabilities are generated after refitting on full validation split.
            p_test_cal = apply_calibration(method, p_valid_base, y_valid.values, p_test_base)

            calibration_rows.append(
                {
                    "model": model_name,
                    "method": method,
                    "cal_logloss": eval_m["logloss"],
                    "cal_ece": expected_calibration_error(y_eval, p_eval_cal),
                    "cal_auc": eval_m["auc"],
                    "cal_f1": eval_m["f1"],
                }
            )
            calibrated_test_probs[(model_name, method)] = p_test_cal
        except Exception as e:
            print(f"{model_name}: {method} skipped ({e})")

calibration_df = pd.DataFrame(calibration_rows)
calibration_df["logloss_rank"] = calibration_df["cal_logloss"].rank(method="min")
calibration_df["ece_rank"] = calibration_df["cal_ece"].rank(method="min")
calibration_df["avg_rank"] = (calibration_df["logloss_rank"] + calibration_df["ece_rank"]) / 2.0
calibration_df = calibration_df.sort_values(["avg_rank", "cal_logloss", "cal_ece"]).reset_index(drop=True)

calibration_path = ARTIFACT_DIR / "calibration_results_2015_rebuild.csv"
calibration_df.to_csv(calibration_path, index=False)

calibration_df.head(20)

,model,method,cal_logloss,cal_ece,cal_auc,cal_f1,logloss_rank,ece_rank,avg_rank
0,catboost,base,0.436397,0.011229,0.856096,0.680582,2.0,7.0,4.5
1,catboost,venn_abers,0.439630,0.009929,0.855054,0.675693,8.0,3.0,5.5
2,catboost,isotonic,0.440092,0.009109,0.855003,0.675440,9.0,2.0,5.5
3,lightgbm,venn_abers,0.437495,0.011670,0.856338,0.674382,3.0,9.0,6.0
4,random_forest,base,0.438143,0.011661,0.854977,0.682274,5.0,8.0,6.5
5,lightgbm,isotonic,0.439340,0.010769,0.856275,0.674470,7.0,6.0,6.5
6,lightgbm,base,0.435535,0.012961,0.856818,0.680566,1.0,14.0,7.5
7,xgboost,base,0.437843,0.011874,0.854558,0.672191,4.0,11.0,7.5
8,adaboost,isotonic,0.445287,0.009015,0.850058,0.652889,18.0,1.0,9.5
9,adaboost,venn_abers,0.444390,0.010325,0.850153,0.670841,16.0,4.0,10.0


In [42]:
best_cal = calibration_df.iloc[0]
best_key = (best_cal["model"], best_cal["method"])
best_test_probs = calibrated_test_probs[best_key]
best_model_name = best_cal["model"]
best_method = best_cal["method"]
best_model = final_models[best_model_name]

best_metrics = metric_pack(y_test, best_test_probs, threshold=0.5)
best_summary = {
    "best_model": best_model_name,
    "calibration_method": best_method,
    "selection_cal_logloss": float(best_cal["cal_logloss"]),
    "selection_cal_ece": float(best_cal["cal_ece"]),
    "test_accuracy": best_metrics["accuracy"],
    "test_precision": best_metrics["precision"],
    "test_recall": best_metrics["recall"],
    "test_f1": best_metrics["f1"],
    "test_auc": best_metrics["auc"],
    "test_logloss": best_metrics["logloss"],
    "test_ece": expected_calibration_error(y_test.values, best_test_probs),
}

summary_path = ARTIFACT_DIR / "best_calibrated_summary_2015_rebuild.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(best_summary, f, indent=2)

print("Best calibrated configuration (selected on validation holdout):")
print(pd.DataFrame([best_summary]).to_string(index=False))
print(f"Saved: {calibration_path}")
print(f"Saved: {summary_path}")

Best calibrated configuration (selected on validation holdout):
best_model calibration_method  selection_cal_logloss  selection_cal_ece  test_accuracy  test_precision  test_recall  test_f1  test_auc  test_logloss  test_ece
  catboost               base               0.436397           0.011229       0.770581        0.652783     0.678384 0.665337  0.845556      0.450667  0.009144
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\calibration_results_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\best_calibrated_summary_2015_rebuild.json


In [55]:
# Full calibration sweep across all Stage-2 retained candidates (expected: 96 candidates x 4 methods = 384 rows).
if 'stage2_best2_df' not in globals() or stage2_best2_df is None or len(stage2_best2_df) == 0:
    raise RuntimeError('stage2_best2_df is empty. Run the Stage-2 best-per-balance cell first.')

candidate_df = stage2_best2_df.copy()
candidate_df = candidate_df[candidate_df['params'].apply(lambda p: isinstance(p, dict))].reset_index(drop=True)
candidate_df['candidate_id'] = np.arange(1, len(candidate_df) + 1)

if len(candidate_df) == 0:
    raise RuntimeError('No valid candidate parameter dictionaries found in stage2_best2_df.')

# Keep fixed methods so row count is deterministic: base + isotonic + platt + venn_abers.
calibration_methods = ['base', 'isotonic', 'platt', 'venn_abers']
expected_rows = int(len(candidate_df) * len(calibration_methods))

# Lower epoch budget than final selection to keep this comprehensive sweep tractable.
CALIBRATION_CANDIDATE_EPOCHS = STAGE2_EPOCHS

all_rows = []

for i, cand in candidate_df.iterrows():
    model_name = str(cand['model'])
    params = cand['params']
    strategy = get_balance_strategy(params)

    rank_within_strategy = np.nan
    if 'rank_within_balance_strategy' in candidate_df.columns and pd.notna(cand.get('rank_within_balance_strategy', np.nan)):
        rank_within_strategy = int(cand['rank_within_balance_strategy'])

    trial_value = np.nan
    if 'trial' in candidate_df.columns and pd.notna(cand.get('trial', np.nan)):
        trial_value = int(cand['trial'])

    base_row = {
        'candidate_id': int(cand['candidate_id']),
        'model': model_name,
        'balance_strategy': strategy,
        'rank_within_balance_strategy': rank_within_strategy,
        'trial': trial_value,
        'candidate_stage2_auc_mean': float(cand['auc_mean']) if 'auc_mean' in candidate_df.columns and pd.notna(cand.get('auc_mean', np.nan)) else np.nan,
        'candidate_stage2_logloss_mean': float(cand['logloss_mean']) if 'logloss_mean' in candidate_df.columns and pd.notna(cand.get('logloss_mean', np.nan)) else np.nan,
        'candidate_stage2_f1_mean': float(cand['f1_mean']) if 'f1_mean' in candidate_df.columns and pd.notna(cand.get('f1_mean', np.nan)) else np.nan,
        'candidate_stage2_score': float(cand['stage_score']) if 'stage_score' in candidate_df.columns and pd.notna(cand.get('stage_score', np.nan)) else np.nan,
        'params_json': json.dumps(strip_internal_params(params), sort_keys=True, default=str),
    }

    try:
        X_train_fit, y_train_fit, sample_weight = prepare_training_data(
            model_name,
            X_train_final,
            y_train,
            strategy,
        )
        model = build_model(
            model_name,
            params,
            epoch_budget=CALIBRATION_CANDIDATE_EPOCHS,
            balance_strategy=strategy,
            y_train_ref=y_train_fit,
        )
        model = fit_model(
            model,
            X_train_fit,
            y_train_fit,
            X_valid_final,
            y_valid,
            early_stopping_rounds=10 if CALIBRATION_CANDIDATE_EPOCHS >= STAGE2_EPOCHS else None,
            sample_weight=sample_weight,
        )

        p_valid_base = safe_predict_proba(model, X_valid_final)
        p_test_base = safe_predict_proba(model, X_test_final)
        p_fit, p_eval, y_fit, y_eval = train_test_split(
            p_valid_base,
            y_valid.values,
            test_size=0.50,
            random_state=RANDOM_SEED,
            stratify=y_valid.values,
        )
        fit_error = None
    except Exception as e:
        fit_error = f'fit_error: {e}'

    for method in calibration_methods:
        row = dict(base_row)
        row.update(
            {
                'calibration_method': method,
                'calibration_accuracy': np.nan,
                'calibration_precision': np.nan,
                'calibration_recall': np.nan,
                'calibration_f1': np.nan,
                'calibration_auc': np.nan,
                'calibration_logloss': np.nan,
                'calibration_ece': np.nan,
                'test_accuracy_calibrated': np.nan,
                'test_precision_calibrated': np.nan,
                'test_recall_calibrated': np.nan,
                'test_f1_calibrated': np.nan,
                'test_auc_calibrated': np.nan,
                'test_logloss_calibrated': np.nan,
                'test_ece_calibrated': np.nan,
                'error': None,
            }
        )

        if fit_error is not None:
            row['error'] = fit_error
            all_rows.append(row)
            continue

        try:
            p_eval_cal = apply_calibration(method, p_fit, y_fit, p_eval)
            eval_met = metric_pack(y_eval, p_eval_cal, threshold=0.5)

            p_test_cal = apply_calibration(method, p_valid_base, y_valid.values, p_test_base)
            test_met = metric_pack(y_test, p_test_cal, threshold=0.5)

            row.update(
                {
                    'calibration_accuracy': eval_met['accuracy'],
                    'calibration_precision': eval_met['precision'],
                    'calibration_recall': eval_met['recall'],
                    'calibration_f1': eval_met['f1'],
                    'calibration_auc': eval_met['auc'],
                    'calibration_logloss': eval_met['logloss'],
                    'calibration_ece': expected_calibration_error(y_eval, p_eval_cal),
                    'test_accuracy_calibrated': test_met['accuracy'],
                    'test_precision_calibrated': test_met['precision'],
                    'test_recall_calibrated': test_met['recall'],
                    'test_f1_calibrated': test_met['f1'],
                    'test_auc_calibrated': test_met['auc'],
                    'test_logloss_calibrated': test_met['logloss'],
                    'test_ece_calibrated': expected_calibration_error(y_test.values, p_test_cal),
                }
            )
        except Exception as e:
            row['error'] = str(e)

        all_rows.append(row)

    if (i + 1) % 5 == 0 or (i + 1) == len(candidate_df):
        print(f'  calibrated candidates {i + 1}/{len(candidate_df)} -> rows {len(all_rows)}/{expected_rows}')

all_calibration_df = pd.DataFrame(all_rows)
ok_mask = all_calibration_df['error'].isna()

all_calibration_df['calibration_logloss_rank'] = np.nan
all_calibration_df['calibration_ece_rank'] = np.nan
all_calibration_df['test_logloss_rank'] = np.nan
all_calibration_df['test_auc_rank'] = np.nan
all_calibration_df['test_f1_rank'] = np.nan
all_calibration_df['composite_rank'] = np.nan

if ok_mask.any():
    all_calibration_df.loc[ok_mask, 'calibration_logloss_rank'] = all_calibration_df.loc[ok_mask, 'calibration_logloss'].rank(method='min', ascending=True)
    all_calibration_df.loc[ok_mask, 'calibration_ece_rank'] = all_calibration_df.loc[ok_mask, 'calibration_ece'].rank(method='min', ascending=True)
    all_calibration_df.loc[ok_mask, 'test_logloss_rank'] = all_calibration_df.loc[ok_mask, 'test_logloss_calibrated'].rank(method='min', ascending=True)
    all_calibration_df.loc[ok_mask, 'test_auc_rank'] = all_calibration_df.loc[ok_mask, 'test_auc_calibrated'].rank(method='min', ascending=False)
    all_calibration_df.loc[ok_mask, 'test_f1_rank'] = all_calibration_df.loc[ok_mask, 'test_f1_calibrated'].rank(method='min', ascending=False)
    all_calibration_df.loc[ok_mask, 'composite_rank'] = (
        all_calibration_df.loc[ok_mask, 'calibration_logloss_rank']
        + all_calibration_df.loc[ok_mask, 'calibration_ece_rank']
        + all_calibration_df.loc[ok_mask, 'test_logloss_rank']
        + all_calibration_df.loc[ok_mask, 'test_auc_rank']
        + all_calibration_df.loc[ok_mask, 'test_f1_rank']
    ) / 5.0

all_calibration_df = all_calibration_df.sort_values(
    ['composite_rank', 'test_auc_calibrated', 'test_f1_calibrated'],
    ascending=[True, False, False],
    na_position='last',
).reset_index(drop=True)

all_calibration_path = ARTIFACT_DIR / 'all_models_calibrated_metrics_2015_rebuild.csv'
try:
    all_calibration_df.to_csv(all_calibration_path, index=False)
except PermissionError:
    ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    all_calibration_path = ARTIFACT_DIR / f'all_models_calibrated_metrics_2015_rebuild_{ts}.csv'
    all_calibration_df.to_csv(all_calibration_path, index=False)

print(f'Saved: {all_calibration_path}')
print(f'Candidates processed: {len(candidate_df)}')
print(f'Expected rows: {expected_rows}')
print(f'Rows: {len(all_calibration_df)}')
print(f'Successful rows: {int(ok_mask.sum())}')
print(f'Failed rows: {int((~ok_mask).sum())}')

top20_cols = [
    'candidate_id',
    'model',
    'balance_strategy',
    'calibration_method',
    'calibration_logloss',
    'calibration_ece',
    'test_logloss_calibrated',
    'test_ece_calibrated',
    'test_auc_calibrated',
    'test_f1_calibrated',
    'test_accuracy_calibrated',
    'test_precision_calibrated',
    'test_recall_calibrated',
    'composite_rank',
    'error',
]

all_calibration_df[top20_cols].head(20)

  calibrated candidates 5/96 -> rows 20/384
  calibrated candidates 10/96 -> rows 40/384
  calibrated candidates 15/96 -> rows 60/384
  calibrated candidates 20/96 -> rows 80/384
  calibrated candidates 25/96 -> rows 100/384
  calibrated candidates 30/96 -> rows 120/384
  calibrated candidates 35/96 -> rows 140/384
  calibrated candidates 40/96 -> rows 160/384
  calibrated candidates 45/96 -> rows 180/384
  calibrated candidates 50/96 -> rows 200/384
  calibrated candidates 55/96 -> rows 220/384
  calibrated candidates 60/96 -> rows 240/384
  calibrated candidates 65/96 -> rows 260/384
  calibrated candidates 70/96 -> rows 280/384
  calibrated candidates 75/96 -> rows 300/384
  calibrated candidates 80/96 -> rows 320/384
  calibrated candidates 85/96 -> rows 340/384
  calibrated candidates 90/96 -> rows 360/384
  calibrated candidates 95/96 -> rows 380/384
  calibrated candidates 96/96 -> rows 384/384
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\all_models_cal

,candidate_id,model,balance_strategy,calibration_method,calibration_logloss,calibration_ece,test_logloss_calibrated,test_ece_calibrated,test_auc_calibrated,test_f1_calibrated,test_accuracy_calibrated,test_precision_calibrated,test_recall_calibrated,composite_rank,error
0,55,xgboost,classweights,venn_abers,0.438356,0.010511,0.451918,0.011993,0.845242,0.673064,0.770140,0.644874,0.703830,27.8,None
1,55,xgboost,classweights,isotonic,0.439054,0.009794,0.451336,0.013342,0.845098,0.672898,0.770096,0.644901,0.703437,29.4,None
2,54,xgboost,base,base,0.437596,0.009159,0.450495,0.009441,0.845304,0.664820,0.770581,0.653247,0.676810,32.6,None
3,53,xgboost,base,venn_abers,0.440408,0.010268,0.451359,0.012990,0.844647,0.675825,0.768993,0.639686,0.716291,36.0,None
4,90,lightgbm,base,isotonic,0.439965,0.011533,0.451819,0.012797,0.845217,0.673198,0.769082,0.642066,0.707503,39.4,None
5,66,catboost,base,base,0.436813,0.009508,0.451143,0.008666,0.845159,0.663577,0.769522,0.651460,0.676154,39.6,None
6,94,lightgbm,smote,venn_abers,0.439016,0.011205,0.452125,0.011271,0.844579,0.672714,0.769302,0.643028,0.705273,40.4,None
7,90,lightgbm,base,base,0.436168,0.010731,0.450405,0.009000,0.845377,0.662359,0.769567,0.652661,0.672350,41.0,None
8,89,lightgbm,base,base,0.436800,0.010723,0.450571,0.011109,0.845292,0.663259,0.770537,0.654534,0.672219,41.0,None
9,65,catboost,base,base,0.437416,0.009530,0.450894,0.009371,0.845492,0.661868,0.768023,0.648897,0.675367,41.6,None


In [43]:
stack_base_pool = final_results_df.sort_values(["valid_stage_score", "valid_auc"], ascending=False)["model"].tolist()
stack_members = stack_base_pool[: min(STACK_TOP_BASE_MODELS, len(stack_base_pool))]
if len(stack_members) < 2:
    raise RuntimeError("Need at least 2 trained models for stacking optimization.")

print("Stacking base models:", stack_members)


def build_stack_feature_matrices(member_names, n_splits=5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    y_train_arr = np.asarray(y_train).astype(int)

    P_train_oof = np.zeros((len(X_train_final), len(member_names)))
    P_valid_meta = np.zeros((len(X_valid_final), len(member_names)))
    P_test_meta = np.zeros((len(X_test_final), len(member_names)))

    for col_idx, model_name in enumerate(member_names):
        if model_name not in selected_final_params:
            raise RuntimeError(f"Missing selected params for stack member: {model_name}")

        params = selected_final_params[model_name]
        strategy = get_balance_strategy(params)
        valid_fold_preds = []
        test_fold_preds = []
        print(f"  OOF stack features -> {model_name} ({strategy})")

        for _, (tr_idx, va_idx) in enumerate(splitter.split(X_train_final, y_train_arr), start=1):
            Xtr = X_train_final.iloc[tr_idx]
            Xva = X_train_final.iloc[va_idx]
            ytr = y_train.iloc[tr_idx]
            yva = y_train.iloc[va_idx]

            Xtr_fit, ytr_fit, sample_weight = prepare_training_data(model_name, Xtr, ytr, strategy)
            model = build_model(
                model_name,
                params,
                epoch_budget=FINAL_EPOCHS,
                balance_strategy=strategy,
                y_train_ref=ytr_fit,
            )
            model = fit_model(
                model,
                Xtr_fit,
                ytr_fit,
                Xva,
                yva,
                early_stopping_rounds=15,
                sample_weight=sample_weight,
            )

            P_train_oof[va_idx, col_idx] = safe_predict_proba(model, Xva)
            valid_fold_preds.append(safe_predict_proba(model, X_valid_final))
            test_fold_preds.append(safe_predict_proba(model, X_test_final))

        P_valid_meta[:, col_idx] = np.mean(np.column_stack(valid_fold_preds), axis=1)
        P_test_meta[:, col_idx] = np.mean(np.column_stack(test_fold_preds), axis=1)

    return P_train_oof, P_valid_meta, P_test_meta


P_train_oof, P_valid_meta, P_test_meta = build_stack_feature_matrices(
    stack_members,
    n_splits=STACK_OOF_FOLDS,
)

meta_space = {
    "C": loguniform(1e-4, 1e3),
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "class_weight": [None, "balanced"],
}
meta_trials = list(ParameterSampler(meta_space, n_iter=STACK_META_TRIALS, random_state=RANDOM_SEED))

meta_rows = []
best_meta_model = None
best_meta_params = None
best_meta_valid_probs = None
best_meta_test_probs = None
best_meta_score = -np.inf

for i, params in enumerate(meta_trials, start=1):
    try:
        penalty = params["penalty"]
        solver = params["solver"]
        if penalty == "l1" and solver not in {"liblinear", "saga"}:
            solver = "liblinear"

        meta_model = LogisticRegression(
            C=float(params["C"]),
            penalty=penalty,
            solver=solver,
            class_weight=params.get("class_weight", None),
            max_iter=8000,
            random_state=RANDOM_SEED,
        )
        meta_model.fit(P_train_oof, y_train.values)

        p_valid = np.clip(meta_model.predict_proba(P_valid_meta)[:, 1], 1e-6, 1 - 1e-6)
        valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
        valid_score = stage_score(valid_met)

        meta_rows.append(
            {
                "strategy": "meta_logistic",
                "trial": i,
                "params": params,
                "valid_auc": valid_met["auc"],
                "valid_logloss": valid_met["logloss"],
                "valid_f1": valid_met["f1"],
                "valid_stage_score": valid_score,
            }
        )

        if valid_score > best_meta_score:
            best_meta_score = valid_score
            best_meta_model = meta_model
            best_meta_params = params
            best_meta_valid_probs = p_valid
            best_meta_test_probs = np.clip(meta_model.predict_proba(P_test_meta)[:, 1], 1e-6, 1 - 1e-6)
    except Exception as e:
        meta_rows.append({"strategy": "meta_logistic", "trial": i, "error": str(e), "valid_stage_score": -999.0})

    if i % 25 == 0 or i == len(meta_trials):
        print(f"  meta search progress: {i}/{len(meta_trials)}")

rng = np.random.default_rng(RANDOM_SEED)
blend_rows = []
best_blend_weights = None
best_blend_valid_probs = None
best_blend_test_probs = None
best_blend_score = -np.inf

for i in range(STACK_BLEND_TRIALS):
    weights = rng.dirichlet(np.ones(len(stack_members)))
    p_valid = np.clip(P_valid_meta @ weights, 1e-6, 1 - 1e-6)
    valid_met = metric_pack(y_valid, p_valid, threshold=0.5)
    valid_score = stage_score(valid_met)

    blend_rows.append(
        {
            "strategy": "weighted_blend",
            "trial": i + 1,
            "weights_json": json.dumps(weights.tolist()),
            "valid_auc": valid_met["auc"],
            "valid_logloss": valid_met["logloss"],
            "valid_f1": valid_met["f1"],
            "valid_stage_score": valid_score,
        }
    )

    if valid_score > best_blend_score:
        best_blend_score = valid_score
        best_blend_weights = weights
        best_blend_valid_probs = p_valid
        best_blend_test_probs = np.clip(P_test_meta @ weights, 1e-6, 1 - 1e-6)

    if (i + 1) % 500 == 0 or (i + 1) == STACK_BLEND_TRIALS:
        print(f"  blend search progress: {i + 1}/{STACK_BLEND_TRIALS}")

if best_meta_valid_probs is None and best_blend_valid_probs is None:
    raise RuntimeError("Stack optimization failed: no valid strategy found.")

if best_meta_valid_probs is not None and (best_meta_score >= best_blend_score):
    stack_strategy = "meta_logistic"
    stack_strategy_details = best_meta_params
    p_stack_valid = best_meta_valid_probs
    p_stack_test = best_meta_test_probs
else:
    stack_strategy = "weighted_blend"
    stack_strategy_details = {"weights": best_blend_weights.tolist()}
    p_stack_valid = best_blend_valid_probs
    p_stack_test = best_blend_test_probs

print(f"Best raw stack strategy selected on validation: {stack_strategy}")

stack_search_df = pd.concat([pd.DataFrame(meta_rows), pd.DataFrame(blend_rows)], ignore_index=True)
stack_search_df = stack_search_df.sort_values("valid_stage_score", ascending=False).reset_index(drop=True)
stack_search_path = ARTIFACT_DIR / "stack_search_results_2015_rebuild.csv"
stack_search_df.to_csv(stack_search_path, index=False)

methods = ["base", "platt", "isotonic"]
if venn_available:
    methods.append("venn_abers")

p_fit, p_eval, y_fit, y_eval = train_test_split(
    p_stack_valid,
    y_valid.values,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_valid.values,
 )

stack_cal_rows = []
stack_test_probs = {}

for method in methods:
    try:
        p_eval_cal = apply_calibration(method, p_fit, y_fit, p_eval)
        eval_m = metric_pack(y_eval, p_eval_cal, threshold=0.5)

        p_test_cal = apply_calibration(method, p_stack_valid, y_valid.values, p_stack_test)
        stack_test_probs[method] = p_test_cal

        stack_cal_rows.append(
            {
                "model": "stack_meta",
                "strategy": stack_strategy,
                "method": method,
                "cal_logloss": eval_m["logloss"],
                "cal_ece": expected_calibration_error(y_eval, p_eval_cal),
                "cal_auc": eval_m["auc"],
                "cal_f1": eval_m["f1"],
            }
        )
    except Exception as e:
        print(f"stack_meta {method} skipped: {e}")

stack_cal_df = pd.DataFrame(stack_cal_rows)
stack_cal_df["logloss_rank"] = stack_cal_df["cal_logloss"].rank(method="min")
stack_cal_df["ece_rank"] = stack_cal_df["cal_ece"].rank(method="min")
stack_cal_df["avg_rank"] = (stack_cal_df["logloss_rank"] + stack_cal_df["ece_rank"]) / 2.0
stack_cal_df = stack_cal_df.sort_values(["avg_rank", "cal_logloss", "cal_ece"]).reset_index(drop=True)

best_stack_method = stack_cal_df.iloc[0]["method"]
best_stack_test_probs = stack_test_probs[best_stack_method]
best_stack_metrics = metric_pack(y_test, best_stack_test_probs, threshold=0.5)
best_stack_metrics["test_ece"] = expected_calibration_error(y_test.values, best_stack_test_probs)

print("Best stack calibration method (selected on validation holdout):", best_stack_method)
print("Best stack test metrics:")
print(pd.DataFrame([best_stack_metrics]).to_string(index=False))

stack_cal_path = ARTIFACT_DIR / "stack_calibration_results_2015_rebuild.csv"
stack_cal_df.to_csv(stack_cal_path, index=False)
print(f"Saved stack search results: {stack_search_path}")
stack_cal_df

Stacking base models: ['lightgbm', 'catboost', 'xgboost', 'random_forest', 'log_reg', 'knn', 'adaboost']
  OOF stack features -> lightgbm (base)
  OOF stack features -> catboost (base)
  OOF stack features -> xgboost (base)
  OOF stack features -> random_forest (base)
  OOF stack features -> log_reg (base)
  OOF stack features -> knn (base)
  OOF stack features -> adaboost (base)
  meta search progress: 25/400
  meta search progress: 50/400
  meta search progress: 75/400
  meta search progress: 100/400
  meta search progress: 125/400
  meta search progress: 150/400
  meta search progress: 175/400
  meta search progress: 200/400
  meta search progress: 225/400
  meta search progress: 250/400
  meta search progress: 275/400
  meta search progress: 300/400
  meta search progress: 325/400
  meta search progress: 350/400
  meta search progress: 375/400
  meta search progress: 400/400
  blend search progress: 500/7000
  blend search progress: 1000/7000
  blend search progress: 1500/7000
  bl

,model,strategy,method,cal_logloss,cal_ece,cal_auc,cal_f1,logloss_rank,ece_rank,avg_rank
0,stack_meta,weighted_blend,base,0.435610,0.012755,0.856672,0.680444,1.0,2.0,1.5
1,stack_meta,weighted_blend,isotonic,0.440571,0.011098,0.856202,0.684358,3.0,1.0,2.0
2,stack_meta,weighted_blend,venn_abers,0.440389,0.013456,0.856356,0.684772,2.0,3.0,2.5
3,stack_meta,weighted_blend,platt,0.443049,0.035990,0.856672,0.672902,4.0,4.0,4.0


In [44]:
if not shap_available:
    print("SHAP skipped: package not installed.")
else:
    def is_tree_model(model):
        name = type(model).__name__.lower()
        return any(tok in name for tok in ["xgb", "lgb", "catboost", "forest", "boost", "tree"])

    explain_model_name = best_model_name if is_tree_model(best_model) else None
    if explain_model_name is None:
        for k, m in final_models.items():
            if is_tree_model(m):
                explain_model_name = k
                break

    if explain_model_name is None:
        print("SHAP skipped: no tree-based model available.")
    else:
        explain_model = final_models[explain_model_name]
        X_bg = X_train_final.sample(min(2000, len(X_train_final)), random_state=RANDOM_SEED)

        try:
            explainer = shap.TreeExplainer(explain_model)
            shap_values = explainer.shap_values(X_bg)
            if isinstance(shap_values, list):
                sv = shap_values[1] if len(shap_values) > 1 else shap_values[0]
            else:
                sv = shap_values

            shap_df = pd.DataFrame(
                {
                    "feature": X_bg.columns,
                    "mean_abs_shap": np.abs(np.asarray(sv)).mean(axis=0),
                }
            ).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

            shap_path = EXPLAIN_DIR / "shap_importance_2015_rebuild.csv"
            shap_df.to_csv(shap_path, index=False)

            print(f"SHAP model used: {explain_model_name}")
            print(shap_df.head(20).to_string(index=False))
            print(f"Saved: {shap_path}")
        except Exception as e:
            print(f"SHAP skipped due to runtime issue: {e}")

SHAP model used: catboost
              feature  mean_abs_shap
             num__age       1.061767
             num__bmi       0.561184
      num__psc_anthro       0.192560
             num__whr       0.119958
  num__regcode_anthro       0.090018
num__fe_alcohol_level       0.057744
             num__fg3       0.040797
        num__mos_preg       0.035049
              num__cu       0.026587
 num__provcode_anthro       0.026016
num__fe_smoking_level       0.017101
             num__fg9       0.016825
             num__fg6       0.015564
   num__mos_lactation       0.015000
    num__anthro_group       0.014403
       num__Total_Fat       0.013895
            num__fg19       0.013606
            num__fg11       0.012964
            num__fg16       0.011137
             num__fg4       0.010920
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\explanations\shap_importance_2015_rebuild.csv


In [45]:
if not lime_available:
    print("LIME skipped: package not installed.")
else:
    lime_model = best_model
    lime_model_name = best_model_name

    lime_explainer = LimeTabularExplainer(
        training_data=X_train_final.values,
        feature_names=X_train_final.columns.tolist(),
        class_names=["class_0", "class_1"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_SEED,
    )

    lime_rows = []
    explain_cases = min(5, len(X_test_final))

    for i in range(explain_cases):
        instance = X_test_final.iloc[i].values
        exp = lime_explainer.explain_instance(
            instance,
            lime_model.predict_proba,
            num_features=min(20, X_train_final.shape[1]),
        )

        html_path = LIME_DIR / f"lime_case_{i + 1}.html"
        exp.save_to_file(str(html_path))

        for rule, weight in exp.as_list():
            lime_rows.append(
                {
                    "case_number": i + 1,
                    "model": lime_model_name,
                    "feature_rule": rule,
                    "weight": float(weight),
                }
            )

    lime_local_df = pd.DataFrame(lime_rows)
    lime_local_path = EXPLAIN_DIR / "lime_local_weights_2015_rebuild.csv"
    lime_local_df.to_csv(lime_local_path, index=False)

    lime_global_df = (
        lime_local_df.assign(abs_weight=lambda d: d["weight"].abs())
        .groupby("feature_rule", as_index=False)["abs_weight"]
        .mean()
        .rename(columns={"abs_weight": "mean_abs_weight"})
        .sort_values("mean_abs_weight", ascending=False)
        .reset_index(drop=True)
    )
    lime_global_path = EXPLAIN_DIR / "lime_global_importance_2015_rebuild.csv"
    lime_global_df.to_csv(lime_global_path, index=False)

    print("LIME global feature rules:")
    print(lime_global_df.head(20).to_string(index=False))
    print(f"Saved: {lime_local_path}")
    print(f"Saved: {lime_global_path}")
    print(f"Saved HTML explanations in: {LIME_DIR}")

LIME global feature rules:
                        feature_rule  mean_abs_weight
                   num__age <= -0.88         0.313905
                   num__bmi <= -0.82         0.146153
           -0.88 < num__age <= -0.23         0.140516
            -0.23 < num__age <= 0.79         0.101422
           -0.82 < num__bmi <= -0.08         0.081726
     -0.93 < num__psc_anthro <= 1.03         0.069988
            num__psc_anthro <= -0.97         0.058807
-0.64 < num__regcode_anthro <= -0.31         0.026838
          num__psurec_anthro > -0.05         0.026247
                   num__whr <= -0.64         0.024972
                    num__fg3 > -0.24         0.021318
          num__regcode_anthro > 0.12         0.020971
           -0.18 < num__fg10 <= 0.23         0.020380
             num__ethnicity <= -0.26         0.017263
            -0.08 < num__bmi <= 0.64         0.017063
  -0.78 < num__mos_lactation <= 0.08         0.017050
           -0.64 < num__whr <= -0.09         0.016866
 

In [46]:
final_results_path = ARTIFACT_DIR / "final_model_results_2015_rebuild.csv"
final_results_df.to_csv(final_results_path, index=False)

run_manifest = {
    "data_path": str(data_path),
    "target_col": target_col,
    "target_defined_from_bp": bool(TARGET_DEFINED_FROM_BP),
    "bp_leakage_guard_active": bool(BP_LEAKAGE_GUARD_ACTIVE),
    "bp_dropped_columns": BP_DROPPED_COLUMNS,
    "models_trained": list(final_models.keys()),
    "stage1_trials_per_model": STAGE1_TRIALS_PER_MODEL,
    "stage2_refinements_per_top_config": STAGE2_REFINEMENTS_PER_TOP_CONFIG,
    "top_k_stage1": TOP_K_STAGE1,
    "top_k_stage2": TOP_K_STAGE2,
    "epochs": {"stage1": STAGE1_EPOCHS, "stage2": STAGE2_EPOCHS, "final": FINAL_EPOCHS},
    "cv_folds": {"stage1": CV_FOLDS_STAGE1, "stage2": CV_FOLDS_STAGE2, "stack_oof": STACK_OOF_FOLDS},
    "stack_search": {
        "meta_trials": STACK_META_TRIALS,
        "blend_trials": STACK_BLEND_TRIALS,
        "top_base_models": STACK_TOP_BASE_MODELS,
    },
    "best_stack_strategy": stack_strategy if "stack_strategy" in globals() else None,
    "best_stack_calibration": best_stack_method if "best_stack_method" in globals() else None,
    "collinearity_cutoff": COLLINEARITY_CUTOFF,
    "final_feature_count": int(X_train_final.shape[1]),
    "best_model": best_model_name,
    "best_calibration": best_method,
}
manifest_path = ARTIFACT_DIR / "run_manifest_2015_rebuild.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print("Rebuild complete. Key artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name}")

print("\nModel files:")
for p in sorted(MODEL_DIR.glob("*.joblib")):
    print(f"- {p.name}")

Rebuild complete. Key artifacts:
- best_calibrated_summary_2015_rebuild.json
- calibration_results_2015_rebuild.csv
- final_candidate_selection_2015_rebuild.csv
- final_model_results_2015_rebuild.csv
- raw_variable_drop_audit_2015_rebuild.csv
- raw_variable_kept_identifier_like_2015_rebuild.csv
- run_manifest_2015_rebuild.json
- runtime_variable_list_2015_rebuild.csv
- stack_calibration_results_2015_rebuild.csv
- stack_search_results_2015_rebuild.csv
- variable_audit_2015_rebuild.csv
- variable_audit_suspect_kept_2015_rebuild.csv

Model files:
- adaboost.joblib
- catboost.joblib
- knn.joblib
- lightgbm.joblib
- log_reg.joblib
- naive_bayes.joblib
- random_forest.joblib
- xgboost.joblib


In [47]:
# Fresh variable/identifier audit (independent cell, dictionary-aware).
def is_id_like_var(name):
    n = str(name).lower()
    return any(tok in n for tok in ["id", "subject", "participant", "seqn", "identifier", "record", "uuid", "case", "sample", "member_code", "hhnum"] )


def is_bp_like_var(name):
    n = str(name).lower()
    return any(tok in n for tok in ["ave_sbp", "ave_dbp", "sbp", "dbp", "systolic", "diastolic", "sysbp", "diabp", "blood_pressure"] )


dict_drop_set = set(globals().get("DICT_IDENTIFIER_DROPPED_COLUMNS", []))
base_cols = [c for c in df.columns if c != target_col]
train_base = X.loc[y_train.index]
train_uniq_ratio = {c: train_base[c].nunique(dropna=False) / max(len(train_base), 1) for c in train_base.columns}

rows = []
for col in base_cols:
    rows.append(
        {
            "stage": "raw_base",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": train_uniq_ratio.get(col, np.nan),
            "dropped_bp_guard": col in BP_DROPPED_COLUMNS,
            "dropped_dict_rule": col in dict_drop_set,
            "dropped_id_rule": col in drop_id_like,
            "kept_after_base_filters": col in X_train.columns,
        }
    )

for col in sorted(set(X_train_fe.columns) - set(X_train.columns)):
    rows.append(
        {
            "stage": "engineered",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
        }
    )

for col in X_train_final.columns:
    rows.append(
        {
            "stage": "final_model",
            "variable": col,
            "identifier_like": is_id_like_var(col),
            "bp_like": is_bp_like_var(col),
            "train_nunique_ratio": np.nan,
            "dropped_bp_guard": False,
            "dropped_dict_rule": False,
            "dropped_id_rule": False,
            "kept_after_base_filters": True,
        }
    )

var_list_df = pd.DataFrame(rows).sort_values(["stage", "variable"]).reset_index(drop=True)
final_suspect_df = var_list_df[(var_list_df["stage"] == "final_model") & (var_list_df["identifier_like"] | var_list_df["bp_like"])].copy()
runtime_var_names_df = pd.DataFrame({"name": sorted([n for n in globals().keys() if not n.startswith("_")])})

var_list_path = ARTIFACT_DIR / "variable_list_2015_rebuild.csv"
suspect_path = ARTIFACT_DIR / "variable_list_suspect_final_2015_rebuild.csv"
runtime_names_path = ARTIFACT_DIR / "runtime_variable_names_2015_rebuild.csv"

var_list_df.to_csv(var_list_path, index=False)
final_suspect_df.to_csv(suspect_path, index=False)
runtime_var_names_df.to_csv(runtime_names_path, index=False)

print(f"Saved: {var_list_path}")
print(f"Saved: {suspect_path}")
print(f"Saved: {runtime_names_path}")
print(f"Dictionary files scanned: {len(globals().get('DICTIONARY_FILES_USED', []))}")
print(f"Dictionary ID columns dropped: {sorted(list(dict_drop_set))}")
print(var_list_df.groupby("stage")["variable"].count().to_string())
print(f"Potential id/bp-like final features: {len(final_suspect_df)}")
final_suspect_df.head(30)

Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\variable_list_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\variable_list_suspect_final_2015_rebuild.csv
Saved: c:\Jon\College\Thesis\V2.2.1.1\main_2015_balanced_gpu_artifacts\runtime_variable_names_2015_rebuild.csv
Dictionary files scanned: 10
Dictionary ID columns dropped: ['hhnum', 'member_code']
stage
engineered      2
final_model    39
raw_base       80
Potential id/bp-like final features: 0


,stage,variable,identifier_like,bp_like,train_nunique_ratio,dropped_bp_guard,dropped_dict_rule,dropped_id_rule,kept_after_base_filters
